# Train the resume optimizer

This notebook is only the fine-tune. It does not change the website until you download the adapter and load it.

1. Upload this file to [Google Colab](https://colab.research.google.com).
2. Runtime → Change runtime type → **T4 GPU**.
3. Run every cell from top to bottom. Keep the tab open while the train cell is running.
4. When it finishes, download `adapters/ResumeOptimizer-LoRA-v6.zip`.

Do not run this notebook on the Windows PC. That machine has no NVIDIA GPU, and the train cell stops on purpose without one.


In [ ]:
# Install the training libraries. Safe to rerun.
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.51.0",
    "peft",
    "bitsandbytes",
    "accelerate",
    "datasets",
])
print("Training libraries are installed.")


In [ ]:
# ============================================================
# V6-00 — PROFESSIONAL RESUME FINE-TUNE
# Fixes jumble/keyword-stuffing. Train on clean one-page format.
# Runtime: T4 GPU.
#
# Run order on a fresh Colab T4 runtime:
#   1. The install cell at the top of this notebook
#   2. Every remaining cell, top to bottom. Keep the tab open during V6-08.
#   3. Download adapters/ResumeOptimizer-LoRA-v6.zip
# ============================================================

print("=" * 70)
print("RESUME OPTIMIZER  •  V6 PROFESSIONAL-FORMAT FINE-TUNE")
print("=" * 70)
print("""
Goal sections:
  CONTACT
  PROFESSIONAL SUMMARY   (3-4 sentences, no keyword dump)
  TECHNICAL SKILLS       (Languages / Frameworks / Databases / Tools)
  PROFESSIONAL EXPERIENCE
  PROJECTS
  EDUCATION
  CERTIFICATIONS         (only if present)

Anti-goals:
  title line that says 'resume'
  'skilled in a, b, c, d...' summary dumps
  bullets ending with ', leveraging python, oop'
  truncated leftovers like 'I am e'
  junk tokens: Present A real, speed, sensor data, English-as-a-skill
""")


In [ ]:
# ============================================================
# V6-01 — RESTART-SAFE PATHS + GPU CHECK
# ============================================================

import os
import gc
import json
import random
import shutil
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Runtime → Change runtime type → T4 GPU, then rerun V6-00."
    )

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount note:", exc)

if "PROJECT_ROOT" not in globals():
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        PROJECT_ROOT = drive_root / "ResumeOptimizer"
    else:
        PROJECT_ROOT = Path("/content/ResumeOptimizer")
        print("Google Drive is not mounted. Files will be saved in", PROJECT_ROOT)

V6_DATA_DIR = PROJECT_ROOT / "datasets" / "v6"
V6_ADAPTER_DIR = PROJECT_ROOT / "adapters" / "ResumeOptimizer-LoRA-v6"
V6_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "qwen_resume_lora_v6"
V5_ADAPTER_DIR = PROJECT_ROOT / "adapters" / "ResumeOptimizer-LoRA-v5"

for folder in [
    PROJECT_ROOT,
    PROJECT_ROOT / "adapters",
    PROJECT_ROOT / "datasets" / "v6",
    PROJECT_ROOT / "checkpoints" / "qwen_resume_lora_v6",
    PROJECT_ROOT / "configs",
    PROJECT_ROOT / "logs",
    V6_DATA_DIR,
    V6_ADAPTER_DIR,
    V6_CHECKPOINT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

if "BASE_MODEL_NAME" not in globals():
    BASE_MODEL_NAME = "Qwen/Qwen3-4B"

if "install_gc_compat_patch" not in globals():
    def install_gc_compat_patch():
        """Fix Trainer passing offload=/every_n_layers= into older PEFT APIs."""
        def _wrap(fn):
            def _safe(self, *args, **kwargs):
                kwargs.pop("offload", None)
                kwargs.pop("every_n_layers", None)
                try:
                    return fn(self, *args, **kwargs)
                except TypeError:
                    gc_kwargs = kwargs.get("gradient_checkpointing_kwargs")
                    try:
                        if gc_kwargs is not None:
                            return fn(self, gradient_checkpointing_kwargs=gc_kwargs)
                        return fn(self)
                    except TypeError:
                        return fn(self)
            return _safe
        try:
            from transformers.modeling_utils import PreTrainedModel
            if not getattr(PreTrainedModel, "_resumeopt_gc_patched", False):
                PreTrainedModel.gradient_checkpointing_enable = _wrap(
                    PreTrainedModel.gradient_checkpointing_enable
                )
                PreTrainedModel._resumeopt_gc_patched = True
        except Exception as exc:
            print("PreTrainedModel patch skipped:", exc)
        try:
            from peft import PeftModel
            if hasattr(PeftModel, "gradient_checkpointing_enable"):
                if not getattr(PeftModel, "_resumeopt_gc_patched", False):
                    PeftModel.gradient_checkpointing_enable = _wrap(
                        PeftModel.gradient_checkpointing_enable
                    )
                    PeftModel._resumeopt_gc_patched = True
        except Exception as exc:
            print("PeftModel patch skipped:", exc)
        return True

ok = install_gc_compat_patch()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("V5 adapter exists:", (V5_ADAPTER_DIR / "adapter_config.json").exists())
print("V6 adapter dir:", V6_ADAPTER_DIR)
print("GC patch:", ok)


In [ ]:
# ============================================================
# V6-02 — FORMAT RULES + JUMBLE DETECTOR
# ============================================================

import re

REQUIRED_HEADERS = [
    "PROFESSIONAL SUMMARY",
    "TECHNICAL SKILLS",
    "PROFESSIONAL EXPERIENCE",
    "EDUCATION",
]

JUMBLE_MARKERS = [
    ", leveraging ",
    "to strengthen delivery against the target role",
    "output only the resume",
    "strict rules",
    "present a real",
    "i am e",
    "skilled in problem-solving",
]


def is_jumbled_resume(text: str) -> bool:
    blob = (text or "").lower()
    stripped = blob.lstrip()
    if stripped.startswith("resume\n") or stripped.startswith("resume\r"):
        return True
    if re.search(r"\bi am e\b", blob):
        return True
    if blob.count(",") >= 10 and "skilled in" in blob[:800]:
        return True
    if any(m in blob for m in JUMBLE_MARKERS):
        return True
    if "languages: english" in blob or "programming languages : english" in blob:
        return True
    return False


def has_professional_structure(text: str) -> bool:
    up = (text or "").upper()
    hits = sum(1 for h in REQUIRED_HEADERS if h in up)
    return hits >= 3 and len(text.strip()) >= 220


def apply_chat(tokenizer, messages, add_generation_prompt=False):
    kwargs = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return tokenizer.apply_chat_template(
            messages, enable_thinking=False, **kwargs
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


print("jumble detector ready")


In [ ]:
# ============================================================
# V6-03 — GOLD PROFESSIONAL TRAINING EXAMPLES
# These teach the exact layout we want in production.
# User prompt shape matches backend PromptTemplates.full_optimization.
# ============================================================

SYSTEM_V6 = """You are an expert AI Resume Optimizer.
THE CANDIDATE'S EXISTING RESUME IS THE SOLE SOURCE OF TRUTH.
Return ONLY a complete professional one-page resume.
Section order: CONTACT, PROFESSIONAL SUMMARY, TECHNICAL SKILLS, PROFESSIONAL EXPERIENCE, PROJECTS, EDUCATION, CERTIFICATIONS.
Write a 3-4 sentence summary. Never dump a comma-separated keyword list.
Group TECHNICAL SKILLS as Languages / Frameworks / Databases / Tools. Never put soft skills there.
Always include a separate SOFT SKILLS section with: Communication, Collaboration, Problem Solving, Critical Thinking, Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership.
Keep the candidate's real employers, titles, dates, projects, and education.
Do not invent metrics. Do not print a title line that says resume.
Do not append 'leveraging <keywords>' to bullets. Do not truncate sentences."""


def v6_user(company, role, jd, resume, evidence="N/A", strategy="professional_one_page"):
    return f"""COMPANY:
{company}

TARGET ROLE:
{role}

JOB DESCRIPTION:
{jd}

ORIGINAL RESUME:
{resume}

CANDIDATE EVIDENCE:
{evidence}

SUPPORTED JD KEYWORDS:
None explicitly mapped

STRATEGY:
{strategy}

STRICT RULES:
1. Never invent employers, job titles, dates, degrees, certifications, or numeric metrics.
2. Keep the candidate's real projects and jobs.
3. Add JOB DESCRIPTION skills into the Skills section even if they were not listed before.
4. Rewrite the Professional Summary for this role using JD keywords.
5. Enhance experience and project bullets by weaving in JD keywords.
6. Prefer a one-page professional resume.
7. Use standard sections in this order: CONTACT, PROFESSIONAL SUMMARY, TECHNICAL SKILLS, PROFESSIONAL EXPERIENCE, PROJECTS, EDUCATION, CERTIFICATIONS.
8. Write a 3-4 sentence summary. Do NOT dump a comma-separated keyword list into the summary.
9. Group skills professionally. Do not add junk phrases or split the job title into fake skills.
10. Keep experience/project bullets as real accomplishments. Do not append "leveraging ...".
11. Output ONLY the complete resume. No title line that just says "resume".

TASK:
Rewrite the resume so it is ATS-optimized for this job description: stronger summary, JD skills listed, and keyword-rich experience/projects.

OUTPUT ONLY THE RESUME."""


def v6_example(uid, role, user, assistant, category="full_resume_optimization"):
    return {
        "id": uid,
        "dataset_version": "v6_professional",
        "category": category,
        "role": role,
        "source": "v6_gold_format",
        "safety_label": "truth_preserving",
        "messages": [
            {"role": "system", "content": SYSTEM_V6},
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ],
    }


GOLD = []

GOLD.append(v6_example(
    "v6_gold_intern_01",
    "Software Engineer Intern",
    v6_user(
        "Google",
        "Software Engineer Intern",
        "We look for students who can write Java and Python, use Git, and collaborate on React frontends and REST APIs. Testing and debugging matter.",
        """A passionate Computer Science undergraduate.
Software Engineer Intern at Google, Mountain View, CA, June 2023 - Present.
Developed and maintained software applications using Java, Python, and React.
Assisted in testing and debugging.
Collaborated with cross-functional teams.
Skills: Java, Python, HTML, CSS, MySQL, MongoDB, Git, REST, React.""",
    ),
    """PRIYA SHARMA
priya.sharma@email.com | Mountain View, CA | github.com/priya

PROFESSIONAL SUMMARY
Computer Science undergraduate and Software Engineer Intern at Google. Builds and maintains production-facing features with Java, Python, and React. Experienced in testing, debugging, and collaborating with cross-functional teams on REST APIs and Git-based workflows.

TECHNICAL SKILLS
Languages: Java, Python, HTML, CSS
Frameworks: React
Databases: MySQL, MongoDB
Tools: Git, REST APIs

PROFESSIONAL EXPERIENCE
Software Engineer Intern | Google | Mountain View, CA | June 2023 – Present
• Developed and maintained software applications using Java, Python, and React
• Assisted in testing and debugging to keep releases stable
• Collaborated with cross-functional teams to improve application reliability

EDUCATION
B.S. Computer Science
""",
))

GOLD.append(v6_example(
    "v6_gold_intern_02",
    "Software Engineer",
    v6_user(
        "Microsoft",
        "Software Engineer",
        "C#, Azure, SQL, Git, teamwork.",
        "Rahul interned at a startup. Built a C# API, wrote SQL queries, used Git. Education: B.Tech CSE.",
    ),
    """RAHUL MEHTA
rahul.mehta@email.com

PROFESSIONAL SUMMARY
Software engineer with internship experience building C# APIs and SQL-backed services. Comfortable with Git collaboration and cloud-hosted application work. Seeking a Software Engineer role at Microsoft.

TECHNICAL SKILLS
Languages: C#, SQL
Cloud: Azure
Tools: Git

PROFESSIONAL EXPERIENCE
Software Engineering Intern | Startup | 2024
• Built a C# API used by internal tools
• Wrote SQL queries for reporting and data access
• Used Git for code reviews and version control

EDUCATION
B.Tech, Computer Science and Engineering
""",
))

GOLD.append(v6_example(
    "v6_gold_projects_01",
    "Backend Developer",
    v6_user(
        "Amazon",
        "Backend Developer",
        "Java, Spring, AWS, Docker, REST, SQL.",
        "Projects: Smart Home Controller in Java with REST APIs. Internship at local firm writing SQL. Skills Java, SQL, Git.",
    ),
    """ANANYA RAO
ananya.rao@email.com

PROFESSIONAL SUMMARY
Backend-focused Computer Science student with internship experience writing SQL and a Java project that exposes REST APIs. Interested in Backend Developer work involving Spring, AWS, and Docker.

TECHNICAL SKILLS
Languages: Java, SQL
Frameworks: Spring
Cloud: AWS
Tools: Git, REST APIs, Docker

PROFESSIONAL EXPERIENCE
Software Intern | Local Firm
• Wrote SQL for application data access
• Used Git to collaborate on service changes

PROJECTS
Smart Home Controller
• Built a Java controller that exposes REST APIs for device actions

EDUCATION
B.Tech, Computer Science
""",
))

print("gold examples so far:", len(GOLD))


In [ ]:
# ============================================================
# V6-03b — EXTRA GOLD BANK (intern, fresher, experienced, data)
# Run after V6-03. Extends GOLD.
# ============================================================

GOLD.append(v6_example(
    "v6_gold_fresher_01",
    "Software Engineer",
    v6_user(
        "Infosys",
        "Software Engineer",
        "Java, SQL, Git, problem solving, teamwork.",
        """Kiran Patel, kiran@email.com
Computer Science student. No internship yet.
Projects: Library Manager in Java with MySQL. Portfolio site with HTML, CSS.
Skills: Java, HTML, CSS, MySQL, Git.
Education: B.E. Computer Science, 2025.""",
    ),
    """KIRAN PATEL
kiran@email.com

PROFESSIONAL SUMMARY
Computer Science graduate focused on Java and SQL application work. Built academic projects covering a MySQL-backed Java app and a static web portfolio. Comfortable with Git and looking for a Software Engineer role at Infosys.

TECHNICAL SKILLS
Languages: Java, HTML, CSS, SQL
Databases: MySQL
Tools: Git

PROJECTS
Library Manager
• Built a Java application that stores catalog data in MySQL

Portfolio Website
• Created a personal site with HTML and CSS

EDUCATION
B.E. Computer Science, 2025
""",
))

GOLD.append(v6_example(
    "v6_gold_experienced_01",
    "Software Engineer",
    v6_user(
        "Stripe",
        "Software Engineer",
        "Python, APIs, SQL, testing, collaboration. Nice to have: AWS.",
        """Jane Patel
Email: jane@example.com
EXPERIENCE
Software Engineer, NovaApps — 2022 to Present
- Built REST APIs in Python and FastAPI for an internal tools platform
- Wrote unit tests and improved CI reliability
- Collaborated with product managers on feature delivery
PROJECTS
Inventory Dashboard
- Created a React dashboard that displayed warehouse stock from PostgreSQL
EDUCATION
B.S. Computer Science, State University, 2021
SKILLS
Python, FastAPI, SQL, React, Git""",
    ),
    """JANE PATEL
jane@example.com

PROFESSIONAL SUMMARY
Software Engineer with experience building Python REST APIs and SQL-backed internal tools. Writes tests, uses Git-based collaboration, and ships features with product partners. Interested in Software Engineer work at Stripe.

TECHNICAL SKILLS
Languages: Python, SQL
Frameworks: FastAPI, React
Databases: PostgreSQL
Tools: Git, REST APIs

PROFESSIONAL EXPERIENCE
Software Engineer | NovaApps | 2022 – Present
• Built REST APIs in Python and FastAPI for an internal tools platform
• Wrote unit tests and improved CI reliability
• Collaborated with product managers on feature delivery

PROJECTS
Inventory Dashboard
• Created a React dashboard that displayed warehouse stock from PostgreSQL

EDUCATION
B.S. Computer Science, State University, 2021
""",
))

GOLD.append(v6_example(
    "v6_gold_data_01",
    "Data Analyst",
    v6_user(
        "Deloitte",
        "Data Analyst",
        "SQL, Excel, Python, dashboards, stakeholder communication.",
        """Meera Singh, meera@email.com
Intern, RetailCo, 2024. Wrote SQL reports, Excel dashboards, some Python cleaning.
Education: B.Sc. Statistics.
Skills: SQL, Excel, Python.""",
    ),
    """MEERA SINGH
meera@email.com

PROFESSIONAL SUMMARY
Data analyst intern experienced in SQL reporting, Excel dashboards, and Python data cleaning. Comfortable explaining findings to non-technical stakeholders. Seeking a Data Analyst role at Deloitte.

TECHNICAL SKILLS
Languages: Python, SQL
Tools: Excel

PROFESSIONAL EXPERIENCE
Data Analyst Intern | RetailCo | 2024
• Wrote SQL reports for weekly operations reviews
• Built Excel dashboards for store performance
• Cleaned source files with Python before analysis

EDUCATION
B.Sc. Statistics
""",
))

GOLD.append(v6_example(
    "v6_gold_fullstack_01",
    "Full Stack Developer",
    v6_user(
        "Shopify",
        "Full Stack Developer",
        "JavaScript, React, Node.js, REST APIs, Git, SQL.",
        """Aman Gupta
Intern at WebLabs, Jan 2024 - Jun 2024.
Built React pages and a Node.js REST API. Used PostgreSQL and Git.
Project: Campus Marketplace in React.
B.Tech IT, 2025.
Skills: JavaScript, React, Node.js, HTML, CSS, Git.""",
    ),
    """AMAN GUPTA
aman.gupta@email.com

PROFESSIONAL SUMMARY
Full-stack intern who builds React interfaces and Node.js REST APIs. Uses PostgreSQL and Git in day-to-day development. Interested in Full Stack Developer work at Shopify.

TECHNICAL SKILLS
Languages: JavaScript, HTML, CSS, SQL
Frameworks: React, Node.js
Databases: PostgreSQL
Tools: Git, REST APIs

PROFESSIONAL EXPERIENCE
Software Intern | WebLabs | Jan 2024 – Jun 2024
• Built React pages for internal dashboards
• Implemented a Node.js REST API backed by PostgreSQL
• Used Git for reviews and version control

PROJECTS
Campus Marketplace
• Built a React storefront for campus listings

EDUCATION
B.Tech, Information Technology, 2025
""",
))

GOLD.append(v6_example(
    "v6_gold_ml_intern_01",
    "Machine Learning Intern",
    v6_user(
        "NVIDIA",
        "Machine Learning Intern",
        "Python, PyTorch, SQL, Git. Experience with model training and evaluation.",
        """Neha Joshi
CS student. Project: Image Classifier in Python with PyTorch. Internship at Campus AI Lab writing Python scripts and SQL.
Skills: Python, PyTorch, SQL, Git.
Education: B.S. Computer Science.""",
    ),
    """NEHA JOSHI
neha.joshi@email.com

PROFESSIONAL SUMMARY
Computer Science student with internship experience writing Python and SQL for applied ML work. Built a PyTorch image classifier and is comfortable training and evaluating models with Git-based workflows.

TECHNICAL SKILLS
Languages: Python, SQL
Frameworks: PyTorch
Tools: Git

PROFESSIONAL EXPERIENCE
Research Intern | Campus AI Lab
• Wrote Python scripts for data preparation
• Used SQL to pull labeled examples for experiments

PROJECTS
Image Classifier
• Trained a PyTorch classifier and reported evaluation metrics

EDUCATION
B.S. Computer Science
""",
))

print("gold examples after 03b:", len(GOLD))
for row in GOLD:
    assistant = row["messages"][-1]["content"]
    assert has_professional_structure(assistant), row["id"]
    assert not is_jumbled_resume(assistant), row["id"]
print("03b gold passed format checks")


In [ ]:
# ============================================================
# V6-03c — ANTI-JUMBLE PAIRS (matches the broken Optimize preview)
# Teaches: no 'resume' title, no keyword dump, no leveraging, no 'I am e'
# ============================================================

SCREENSHOT_JUMBLE = """resume
PROFESSIONAL SUMMARY
software engineer targeting google, skilled in problem-solving, python, oop, sql, git, communication, java, teamwork, c++, go, software, engineer. A passionate and dedicated Computer Science undergraduate with a strong academic record and hands-on experience in various technology-driven projects. My problem-solving skills and technical expertise are well-suited for real-world applications, particularly in automation, smart systems, and safety technologies. I am e

TECHNICAL SKILLS
Databases : MySQL, MongoDB, Languages: English, Programming Languages : Java, Python, HTML, CSS, Critical Thinking, Team Collaboration, Effective Communication, Tools & Platforms: Git, REST, Present A real, speed, React, sensor data, or user preferences, improving convenience, security, Arduino, java, spring, sql, git, problem-solving, oop, communication, teamwork, c++, go, software, engineer

PROFESSIONAL EXPERIENCE
Software Engineer Intern
Google, Mountain View, CA
June 2023 - Present
• Developed and maintained software applications using Java, Python, and React, leveraging problem-solving, oop.
• Assisted in testing and debugging code to ensure high-quality software, leveraging sql, communication.
• Collaborated with cross-functional teams to improve software performance and reliability, leveraging teamwork, c++.
- Learned and applied new technologies, including Git and REST APIs.
• Applied react, sql to strengthen delivery against the target role.
"""

SCREENSHOT_CLEAN = """CANDIDATE NAME
email@example.com | Mountain View, CA

PROFESSIONAL SUMMARY
Computer Science undergraduate and Software Engineer Intern at Google. Builds and maintains applications with Java, Python, and React, and works with MySQL, MongoDB, and REST APIs. Collaborates with cross-functional teams on testing, debugging, and reliability improvements.

TECHNICAL SKILLS
Languages: Java, Python, C++, Go, HTML, CSS, SQL
Frameworks: React, Spring
Databases: MySQL, MongoDB
Tools: Git, REST APIs, Arduino

PROFESSIONAL EXPERIENCE
Software Engineer Intern | Google | Mountain View, CA | June 2023 – Present
• Developed and maintained software applications using Java, Python, and React
• Assisted in testing and debugging to keep software quality high
• Collaborated with cross-functional teams to improve performance and reliability
• Learned and applied Git and REST APIs in day-to-day development

EDUCATION
B.S. Computer Science
"""

GOLD.append(v6_example(
    "v6_antijumble_screenshot_01",
    "Software Engineer Intern",
    v6_user(
        "Google",
        "Software Engineer Intern",
        "Students who write Java and Python, use Git, collaborate on React frontends and REST APIs. Testing and debugging matter. SQL is useful.",
        SCREENSHOT_JUMBLE,
    ),
    SCREENSHOT_CLEAN,
    category="anti_jumble",
))

GOLD.append(v6_example(
    "v6_antijumble_screenshot_02",
    "Software Engineer Intern",
    "Rewrite this jumbled resume into a professional one-page resume. Keep facts. Do not print the word resume as a title.\n\n" + SCREENSHOT_JUMBLE,
    SCREENSHOT_CLEAN,
    category="anti_jumble",
))

GOLD.append(v6_example(
    "v6_skills_cleanup_01",
    "Software Engineer Intern",
    "Clean this skills dump into grouped technical skills. Drop English, soft skills, and junk tokens.\n\nDatabases : MySQL, MongoDB, Languages: English, Programming Languages : Java, Python, HTML, CSS, Critical Thinking, Team Collaboration, Effective Communication, Tools & Platforms: Git, REST, Present A real, speed, React, sensor data",
    """TECHNICAL SKILLS
Languages: Java, Python, HTML, CSS
Frameworks: React
Databases: MySQL, MongoDB
Tools: Git, REST APIs
""",
    category="skills_cleanup",
))

GOLD.append(v6_example(
    "v6_bullet_cleanup_01",
    "Software Engineer Intern",
    "Rewrite these bullets. Remove 'leveraging' keyword tails. Keep the real work.\n\n• Developed applications using Java, Python, and React, leveraging problem-solving, oop.\n• Assisted in testing and debugging, leveraging sql, communication.\n• Collaborated with teams, leveraging teamwork, c++.\n• Applied react, sql to strengthen delivery against the target role.",
    """• Developed applications using Java, Python, and React
• Assisted in testing and debugging
• Collaborated with cross-functional teams to improve reliability
""",
    category="bullet_cleanup",
))

GOLD.append(v6_example(
    "v6_summary_cleanup_01",
    "Software Engineer Intern",
    "Rewrite this summary as 3-4 professional sentences. Do not dump keywords.\n\nsoftware engineer targeting google, skilled in problem-solving, python, oop, sql, git, communication, java, teamwork, c++, go, software, engineer. A passionate CS undergraduate. I am e",
    """Computer Science undergraduate targeting a Software Engineer Intern role. Builds software with Java, Python, and related tools, and is comfortable using Git and SQL. Strong collaborator who focuses on clear, working code rather than keyword lists.
""",
    category="summary_cleanup",
))

# Repeat the screenshot pair so format gets more gradient signal.
GOLD.extend([GOLD[-5], GOLD[-4]])

print("gold examples after 03c:", len(GOLD))
print("anti-jumble ids:", [r["id"] for r in GOLD if r["category"] in {"anti_jumble", "skills_cleanup", "bullet_cleanup", "summary_cleanup"}])


In [ ]:
# ============================================================
# V6-03d — SOFT SKILLS COLUMN (never mix into languages/tools)
# Run after V6-03c. Extends GOLD and patches earlier gold resumes.
# ============================================================

SOFT_SKILLS_LINE = (
    "Communication, Collaboration, Problem Solving, Critical Thinking, "
    "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
)
SOFT_SKILLS_BLOCK = "SOFT SKILLS\n" + SOFT_SKILLS_LINE


def ensure_soft_skills_section(text: str) -> str:
    if "SOFT SKILLS" in (text or "").upper():
        return text
    for marker in ["\nPROFESSIONAL EXPERIENCE", "\nPROJECTS", "\nEDUCATION"]:
        if marker in text:
            return text.replace(marker, "\n\n" + SOFT_SKILLS_BLOCK + "\n" + marker, 1)
    return text.rstrip() + "\n\n" + SOFT_SKILLS_BLOCK + "\n"


for row in GOLD:
    ast = row["messages"][-1]["content"]
    row["messages"][-1]["content"] = ensure_soft_skills_section(ast)

GOLD.append(v6_example(
    "v6_soft_split_01",
    "Software Engineer Intern",
    "Separate technical skills from soft skills. Soft skills must not appear under languages, tools, or platforms.\n\nSKILLS: Java, Python, Communication, Git, Teamwork, React, Problem-Solving, MySQL, Critical Thinking, English",
    """CANDIDATE NAME
email@example.com

PROFESSIONAL SUMMARY
Computer Science student who builds applications with Java, Python, and React.

TECHNICAL SKILLS
Languages: Java, Python
Frameworks: React
Databases: MySQL
Tools: Git

SOFT SKILLS
""" + SOFT_SKILLS_LINE + """

EDUCATION
B.S. Computer Science
""",
    category="soft_skills_split",
))

GOLD.append(v6_example(
    "v6_soft_every_role_01",
    "Data Analyst",
    v6_user(
        "Deloitte",
        "Data Analyst",
        "SQL, Excel, communication, teamwork.",
        "Meera interned at RetailCo writing SQL. Skills: SQL, Excel, communication, teamwork.",
    ),
    """MEERA SINGH
meera@email.com

PROFESSIONAL SUMMARY
Data analyst intern experienced in SQL reporting and Excel dashboards.

TECHNICAL SKILLS
Languages: SQL
Tools: Excel

SOFT SKILLS
""" + SOFT_SKILLS_LINE + """

PROFESSIONAL EXPERIENCE
Data Analyst Intern | RetailCo
• Wrote SQL reports for weekly operations reviews

EDUCATION
B.Sc. Statistics
""",
    category="soft_skills_split",
))

print("gold examples after 03d:", len(GOLD))
print("soft-skill gold ready")


In [ ]:
# ============================================================
# V6-03e — MORE ROLES (software, data, ML, cloud, QA, mobile,
# security, business, finance, marketing, sales, HR, design, ops)
# Run after V6-03d. Extends GOLD with one professional example per role.
# ============================================================

if "SOFT_SKILLS_LINE" not in globals():
    SOFT_SKILLS_LINE = (
        "Communication, Collaboration, Problem Solving, Critical Thinking, "
        "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
    )

ROLE_SPECS = [
    dict(role="Frontend Developer", company="Adobe", jd="JavaScript, React, HTML, CSS, Git, accessibility.",
         name="SANA KHAN", email="sana.khan@email.com", title="Frontend Intern", employer="PixelWorks",
         dates="Jan 2024 – Jun 2024", bullets=["Built React pages for an internal dashboard", "Styled layouts with HTML and CSS", "Used Git for code reviews"],
         languages=["JavaScript", "HTML", "CSS"], frameworks=["React"], tools=["Git"],
         education="B.S. Computer Science, 2025"),
    dict(role="Java Developer", company="Infosys", jd="Java, Spring, SQL, Git, REST APIs.",
         name="ARJUN NAIR", email="arjun.nair@email.com", title="Java Intern", employer="CodeNest",
         dates="2024", bullets=["Wrote Java services for internal tools", "Queried application data with SQL", "Used Git and REST APIs"],
         languages=["Java", "SQL"], frameworks=["Spring"], tools=["Git", "REST APIs"],
         education="B.Tech, Computer Science"),
    dict(role="Python Developer", company="JPMorgan", jd="Python, FastAPI, SQL, Git, testing.",
         name="LEILA HASSAN", email="leila.hassan@email.com", title="Python Intern", employer="DataLeaf",
         dates="2024", bullets=["Built Python APIs with FastAPI", "Wrote SQL for reporting", "Added unit tests before merge"],
         languages=["Python", "SQL"], frameworks=["FastAPI"], tools=["Git"],
         education="B.S. Computer Science"),
    dict(role="DevOps Engineer", company="IBM", jd="Linux, Docker, AWS, CI/CD, Git.",
         name="VIKRAM SHAH", email="vikram.shah@email.com", title="DevOps Intern", employer="CloudPark",
         dates="2024", bullets=["Packaged services with Docker", "Helped maintain GitHub Actions pipelines", "Used Linux servers for staging"],
         languages=["Bash"], tools=["Git", "Docker", "AWS", "Linux"],
         education="B.Tech, Information Technology"),
    dict(role="Cloud Engineer", company="Accenture", jd="AWS, Terraform, Linux, Python, Git.",
         name="NINA PATEL", email="nina.patel@email.com", title="Cloud Intern", employer="SkyOps",
         dates="2024", bullets=["Wrote Terraform for a small AWS lab", "Automated setup scripts in Python", "Used Git for infrastructure changes"],
         languages=["Python"], tools=["Git", "AWS", "Linux"],
         education="B.S. Computer Science"),
    dict(role="Site Reliability Engineer", company="Atlassian", jd="Linux, monitoring, Python, incident response, Git.",
         name="OMAR REID", email="omar.reid@email.com", title="SRE Intern", employer="Uptime Labs",
         dates="2024", bullets=["Wrote Python checks for service health", "Documented incident steps for on-call", "Used Linux logs to debug outages"],
         languages=["Python"], tools=["Git", "Linux"],
         education="B.S. Computer Engineering"),
    dict(role="QA Engineer", company="Capgemini", jd="Manual testing, Selenium, Java, Jira, Git.",
         name="RHEA KAPOOR", email="rhea.kapoor@email.com", title="QA Intern", employer="TestHub",
         dates="2024", bullets=["Wrote test cases in Jira", "Automated regression checks with Selenium and Java", "Logged defects before release"],
         languages=["Java"], tools=["Git", "Jira", "Selenium"],
         education="B.E. Computer Science"),
    dict(role="SDET", company="Salesforce", jd="Java, Selenium, API testing, Git, CI.",
         name="BEN PARK", email="ben.park@email.com", title="SDET Intern", employer="QualityForge",
         dates="2024", bullets=["Automated UI checks with Java and Selenium", "Added API tests for checkout flows", "Ran suites in CI before merge"],
         languages=["Java"], tools=["Git", "Selenium"],
         education="B.S. Computer Science"),
    dict(role="Android Developer", company="Samsung", jd="Kotlin, Android, REST APIs, Git.",
         name="ISHAN MEHTA", email="ishan.mehta@email.com", title="Android Intern", employer="AppOrchard",
         dates="2024", bullets=["Built Android screens in Kotlin", "Called REST APIs for feed data", "Used Git for feature branches"],
         languages=["Kotlin"], tools=["Git", "REST APIs"],
         education="B.Tech, Computer Science"),
    dict(role="iOS Developer", company="Apple", jd="Swift, iOS, REST APIs, Git, UIKit.",
         name="MAYA CHEN", email="maya.chen@email.com", title="iOS Intern", employer="Northstar Apps",
         dates="2024", bullets=["Implemented iOS screens in Swift", "Fetched data from REST APIs", "Used Git for pull requests"],
         languages=["Swift"], tools=["Git", "REST APIs"],
         education="B.S. Computer Science"),
    dict(role="Cybersecurity Analyst", company="Deloitte", jd="SIEM, network security, Python, incident triage, Git.",
         name="KAREN LOPEZ", email="karen.lopez@email.com", title="Security Intern", employer="ShieldByte",
         dates="2024", bullets=["Triaged SIEM alerts for a small SOC lab", "Wrote Python scripts to parse logs", "Documented incident notes for review"],
         languages=["Python"], tools=["Git"],
         education="B.S. Cybersecurity"),
    dict(role="Network Engineer", company="Cisco", jd="Routing, switching, TCP/IP, Linux, troubleshooting.",
         name="FARHAN ALI", email="farhan.ali@email.com", title="Network Intern", employer="PacketLine",
         dates="2024", bullets=["Configured lab routers and switches", "Troubleshot TCP/IP connectivity issues", "Used Linux for packet captures"],
         tools=["Linux"], education="B.E. Electronics and Communication"),
    dict(role="Embedded Engineer", company="Bosch", jd="C, C++, microcontrollers, Git, debugging.",
         name="TARA IYER", email="tara.iyer@email.com", title="Embedded Intern", employer="ChipWorks",
         dates="2024", bullets=["Wrote C firmware for a microcontroller lab", "Debugged hardware bring-up issues", "Used Git to version firmware"],
         languages=["C", "C++"], tools=["Git"],
         education="B.Tech, Electronics"),
    dict(role="Data Scientist", company="Nielsen", jd="Python, SQL, statistics, pandas, machine learning.",
         name="ELIAS BROWN", email="elias.brown@email.com", title="Data Science Intern", employer="InsightYard",
         dates="2024", bullets=["Cleaned datasets with Python and pandas", "Wrote SQL to assemble training tables", "Trained a baseline classification model"],
         languages=["Python", "SQL"], education="B.S. Statistics"),
    dict(role="Data Engineer", company="Snowflake", jd="SQL, Python, ETL, Spark, Git, data pipelines.",
         name="PRIYA DAS", email="priya.das@email.com", title="Data Engineering Intern", employer="Pipeworks",
         dates="2024", bullets=["Built SQL transformations for nightly ETL", "Wrote Python jobs to move files", "Used Git for pipeline reviews"],
         languages=["Python", "SQL"], tools=["Git"],
         education="B.Tech, Computer Science"),
    dict(role="Machine Learning Engineer", company="OpenAI", jd="Python, PyTorch, SQL, Git, model evaluation.",
         name="NOAH KIM", email="noah.kim@email.com", title="ML Intern", employer="ModelBay",
         dates="2024", bullets=["Trained a PyTorch model on labeled data", "Evaluated precision and recall", "Stored features with SQL"],
         languages=["Python", "SQL"], frameworks=["PyTorch"], tools=["Git"],
         education="B.S. Computer Science"),
    dict(role="UI/UX Designer", company="Figma", jd="Figma, user research, wireframes, prototyping, accessibility.",
         name="AVERY BROOKS", email="avery.brooks@email.com", title="UX Design Intern", employer="LookLab",
         dates="2024", bullets=["Created wireframes and prototypes in Figma", "Ran five usability interviews", "Documented accessibility notes for engineering"],
         tools=["Figma"], education="B.Des, Interaction Design"),
    dict(role="Product Manager", company="Spotify", jd="roadmaps, user stories, analytics, stakeholder communication, Jira.",
         name="SAMIRA QURESHI", email="samira.qureshi@email.com", title="Product Intern", employer="Launchpad",
         dates="2024", bullets=["Wrote user stories and acceptance criteria in Jira", "Prioritized a small feature roadmap", "Reviewed funnel metrics with analytics"],
         tools=["Jira", "Excel"], education="B.B.A., 2025"),
    dict(role="Business Analyst", company="McKinsey", jd="requirements gathering, process mapping, Excel, SQL, stakeholders.",
         name="DANIEL ORTIZ", email="daniel.ortiz@email.com", title="Business Analyst Intern", employer="OpsLane",
         dates="2024", bullets=["Gathered requirements from operations leads", "Mapped a current-state process", "Built Excel and SQL views for weekly reviews"],
         languages=["SQL"], tools=["Excel"],
         education="B.Com, 2025"),
    dict(role="Financial Analyst", company="Goldman Sachs", jd="Excel, financial modeling, forecasting, reporting, attention to detail.",
         name="HANNAH WELLS", email="hannah.wells@email.com", title="Finance Intern", employer="Ledger & Co",
         dates="2024", bullets=["Built Excel models for monthly forecasts", "Prepared variance reports for leadership", "Checked source figures before publishing"],
         tools=["Excel"], education="B.S. Finance"),
    dict(role="Digital Marketing Specialist", company="HubSpot", jd="SEO, Google Analytics, social media, content, campaign reporting.",
         name="LUCA ROSSI", email="luca.rossi@email.com", title="Marketing Intern", employer="BrightFunnel",
         dates="2024", bullets=["Wrote SEO blog drafts for product pages", "Scheduled social posts and tracked engagement", "Pulled campaign reports from Google Analytics"],
         tools=["Google Analytics", "Excel"],
         education="B.A. Marketing"),
    dict(role="SEO Specialist", company="Semrush", jd="keyword research, on-page SEO, Google Search Console, content briefs.",
         name="JAYA REDDY", email="jaya.reddy@email.com", title="SEO Intern", employer="Ranksmith",
         dates="2024", bullets=["Researched keywords for landing pages", "Wrote on-page briefs for writers", "Monitored Search Console queries"],
         tools=["Google Search Console", "Excel"],
         education="B.A. Mass Communication"),
    dict(role="Sales Executive", company="Oracle", jd="CRM, pipeline management, outreach, negotiation, Excel.",
         name="MATEO SILVA", email="mateo.silva@email.com", title="Sales Development Intern", employer="QuotaTree",
         dates="2024", bullets=["Logged outreach in the CRM", "Qualified inbound leads for account executives", "Tracked pipeline notes in Excel"],
         tools=["Excel"], education="B.B.A."),
    dict(role="HR Coordinator", company="Unilever", jd="recruiting, onboarding, HRIS, scheduling, employee communication.",
         name="FATIMA NOOR", email="fatima.noor@email.com", title="HR Intern", employer="PeopleFirst",
         dates="2024", bullets=["Scheduled interviews and sent candidate updates", "Prepared onboarding checklists", "Maintained employee records in the HRIS"],
         tools=["Excel"], education="B.A. Human Resources"),
    dict(role="Technical Support Engineer", company="Zendesk", jd="troubleshooting, ticketing, SQL, customer communication, Linux.",
         name="OWEN BLAKE", email="owen.blake@email.com", title="Support Intern", employer="Helpstack",
         dates="2024", bullets=["Resolved product tickets in the help desk", "Looked up account data with SQL", "Documented Linux-side reproduction steps"],
         languages=["SQL"], tools=["Linux"],
         education="B.S. Information Systems"),
    dict(role="Technical Writer", company="GitLab", jd="documentation, API reference, Markdown, Git, developer experience.",
         name="CHLOE NGUYEN", email="chloe.nguyen@email.com", title="Docs Intern", employer="WritePath",
         dates="2024", bullets=["Drafted API reference pages in Markdown", "Edited procedure docs with engineering review", "Used Git to publish documentation"],
         tools=["Git"], education="B.A. English"),
    dict(role="Project Coordinator", company="Wipro", jd="schedules, status reporting, Jira, stakeholder updates, Excel.",
         name="ANIKA BOSE", email="anika.bose@email.com", title="Project Intern", employer="Planwell",
         dates="2024", bullets=["Kept Jira boards current for a delivery team", "Sent weekly status reports", "Tracked risks in Excel"],
         tools=["Jira", "Excel"], education="B.B.A."),
    dict(role="Customer Success Associate", company="Twilio", jd="onboarding, retention, CRM, customer communication, Excel.",
         name="LIAM OCONNOR", email="liam.oconnor@email.com", title="Customer Success Intern", employer="Keepwell",
         dates="2024", bullets=["Ran onboarding calls for new accounts", "Logged follow-ups in the CRM", "Tracked usage notes in Excel"],
         tools=["Excel"], education="B.A. Communications"),
    dict(role="Operations Analyst", company="Amazon", jd="process improvement, Excel, SQL, reporting, cross-functional work.",
         name="YARA HADDAD", email="yara.haddad@email.com", title="Operations Intern", employer="Flowline",
         dates="2024", bullets=["Built Excel trackers for warehouse cycle time", "Wrote SQL to audit order exceptions", "Presented a small process-improvement note"],
         languages=["SQL"], tools=["Excel"],
         education="B.S. Operations Management"),
    dict(role="Supply Chain Analyst", company="Nike", jd="inventory, Excel, forecasting, SQL, vendor coordination.",
         name="KENJI MORI", email="kenji.mori@email.com", title="Supply Chain Intern", employer="Routewise",
         dates="2024", bullets=["Updated inventory forecasts in Excel", "Pulled shipment data with SQL", "Coordinated vendor ETAs with operations"],
         languages=["SQL"], tools=["Excel"],
         education="B.S. Supply Chain"),
    dict(role="Graphic Designer", company="Canva", jd="Adobe Illustrator, Photoshop, brand systems, layout, feedback.",
         name="ELENA VASQUEZ", email="elena.vasquez@email.com", title="Design Intern", employer="Ink & Grid",
         dates="2024", bullets=["Designed social and web assets in Illustrator and Photoshop", "Followed a small brand system", "Revised layouts from stakeholder feedback"],
         tools=["Adobe Illustrator", "Photoshop"],
         education="B.F.A. Graphic Design"),
    dict(role="Content Writer", company="Notion", jd="writing, editing, SEO, research, CMS.",
         name="IBRAHIM SALEH", email="ibrahim.saleh@email.com", title="Content Intern", employer="Storymill",
         dates="2024", bullets=["Wrote help-center articles from research briefs", "Edited drafts for tone and SEO", "Published posts in the CMS"],
         tools=["Google Docs"], education="B.A. Journalism"),
    dict(role="Research Analyst", company="Gartner", jd="desk research, Excel, slide reporting, synthesis, stakeholders.",
         name="SOFIA MARTIN", email="sofia.martin@email.com", title="Research Intern", employer="Northline Research",
         dates="2024", bullets=["Completed desk research for a market brief", "Built Excel tables of competitor data", "Drafted slides for the weekly readout"],
         tools=["Excel"], education="B.A. Economics"),
    dict(role="Mechanical Engineer", company="Tesla", jd="CAD, SolidWorks, prototyping, GD&T, testing.",
         name="RAHUL IYENGAR", email="rahul.iyengar@email.com", title="Mechanical Intern", employer="MotionForge",
         dates="2024", bullets=["Modeled parts in SolidWorks", "Built a prototype fixture for bench tests", "Recorded dimensional checks against drawings"],
         tools=["SolidWorks"], education="B.Tech, Mechanical Engineering"),
    dict(role="Electrical Engineer", company="Siemens", jd="circuit design, testing, MATLAB, documentation, safety.",
         name="AMELIE DURAND", email="amelie.durand@email.com", title="Electrical Intern", employer="Voltbench",
         dates="2024", bullets=["Tested circuit boards against a spec", "Analyzed measurements in MATLAB", "Wrote lab notes for the design review"],
         tools=["MATLAB"], education="B.S. Electrical Engineering"),
    dict(role="Civil Engineer", company="AECOM", jd="AutoCAD, site inspection, quantities, reporting, safety.",
         name="JOSE REYES", email="jose.reyes@email.com", title="Civil Intern", employer="Spanfield",
         dates="2024", bullets=["Drafted plan markups in AutoCAD", "Supported site quantity takeoffs", "Wrote daily inspection notes"],
         tools=["AutoCAD"], education="B.E. Civil Engineering"),
]


def _tech_block(spec):
    lines = []
    for label, key in [
        ("Languages", "languages"),
        ("Frameworks", "frameworks"),
        ("Databases", "databases"),
        ("Tools", "tools"),
    ]:
        vals = spec.get(key) or []
        if vals:
            lines.append(f"{label}: " + ", ".join(vals))
    return "\n".join(lines) if lines else "Tools: Excel"


def _orig_resume(spec):
    mixed = []
    for key in ("languages", "frameworks", "databases", "tools"):
        mixed.extend(spec.get(key) or [])
    mixed.extend(["communication", "teamwork", "problem-solving"])
    bullets = "\n".join(f"- {b}" for b in spec["bullets"])
    project = spec.get("project")
    project_block = f"\nPROJECTS\n{project}\n" if project else "\n"
    return f"""{spec['name'].title()}
{spec['email']}
{spec.get('summary_orig') or 'Student with internship experience.'}
EXPERIENCE
{spec['title']} | {spec['employer']} | {spec['dates']}
{bullets}
{project_block}SKILLS: {', '.join(mixed)}
EDUCATION
{spec['education']}
"""


def _assistant(spec):
    bullets = "\n".join(f"• {b}" for b in spec["bullets"])
    project = spec.get("project")
    project_block = f"\n\nPROJECTS\n{project}\n" if project else "\n"
    summary = (
        f"{spec['title']} with hands-on work at {spec['employer']}. "
        f"Uses {', '.join((spec.get('languages') or spec.get('tools') or ['core tools'])[:3])} in day-to-day work. "
        f"Seeking a {spec['role']} role at {spec['company']}."
    )
    return f"""{spec['name']}
{spec['email']}

PROFESSIONAL SUMMARY
{summary}

TECHNICAL SKILLS
{_tech_block(spec)}

SOFT SKILLS
{SOFT_SKILLS_LINE}

PROFESSIONAL EXPERIENCE
{spec['title']} | {spec['employer']} | {spec['dates']}
{bullets}
{project_block}
EDUCATION
{spec['education']}
"""


before = len(GOLD)
for spec in ROLE_SPECS:
    GOLD.append(v6_example(
        "v6_role_" + spec["role"].lower().replace(" ", "_").replace("/", "_"),
        spec["role"],
        v6_user(spec["company"], spec["role"], spec["jd"], _orig_resume(spec)),
        _assistant(spec),
        category="role_coverage",
    ))

roles = sorted({row["role"] for row in GOLD})
print("added role examples:", len(GOLD) - before)
print("gold examples total:", len(GOLD))
print("unique roles:", len(roles))
for role in roles:
    print(" ", role)


In [ ]:
# ============================================================
# V6-03f — MORE ROLES + DENSE ATS KEYWORDS
# Run after V6-03e. Builds extra gold resumes so V6-08 trains
# on many job families and puts JD keywords on the correct lines.
# ============================================================

if "SOFT_SKILLS_LINE" not in globals():
    SOFT_SKILLS_LINE = (
        "Communication, Collaboration, Problem Solving, Critical Thinking, "
        "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
    )

NAMES = [
    ("ALEX RIVERA", "alex.rivera@email.com"),
    ("JORDAN LEE", "jordan.lee@email.com"),
    ("MIRA KAPOOR", "mira.kapoor@email.com"),
    ("CHEN WEI", "chen.wei@email.com"),
    ("SARA ALI", "sara.ali@email.com"),
    ("NOAH PATEL", "noah.patel@email.com"),
    ("ELENA ROSSI", "elena.rossi@email.com"),
    ("KIAN SHAH", "kian.shah@email.com"),
]
EMPLOYERS = ["Nimbus Labs", "HarborTech", "Cedar Systems", "Northwind Apps", "Brightstack", "LumenWorks"]

FAMILY_BANKS = {
    "software": {
        "jd": "Java, Python, JavaScript, TypeScript, SQL, data structures, algorithms, OOP, REST APIs, Git, testing, Agile, system design, debugging.",
        "languages": ["Java", "Python", "JavaScript", "TypeScript", "SQL"],
        "core_cs": ["Data Structures & Algorithms", "Object-Oriented Programming"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "bullets": [
            "Built backend features with Java/Python and REST APIs",
            "Used data structures and algorithms in coding tasks",
            "Wrote tests and used Git in an Agile workflow",
        ],
    },
    "frontend": {
        "jd": "JavaScript, TypeScript, React, HTML5, CSS3, Tailwind, accessibility, REST APIs, Git, responsive design, Next.js.",
        "languages": ["JavaScript", "TypeScript"],
        "frontend": ["React.js", "HTML5/CSS3", "Tailwind CSS"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "bullets": [
            "Built responsive React.js pages with HTML5/CSS3 and Tailwind CSS",
            "Consumed REST APIs and handled loading and error states",
            "Used Git for reviews and kept the UI accessible",
        ],
    },
    "backend": {
        "jd": "Java, Python, Spring Boot, FastAPI, Node.js, REST APIs, SQL, PostgreSQL, Docker, Git, microservices.",
        "languages": ["Java", "Python", "SQL"],
        "backend": ["Spring Boot", "FastAPI", "Node.js", "REST APIs"],
        "databases": ["PostgreSQL"],
        "tools": ["Git", "Docker"],
        "bullets": [
            "Implemented REST APIs with Spring Boot / FastAPI / Node.js",
            "Stored application data in PostgreSQL using SQL",
            "Packaged services with Docker and used Git for reviews",
        ],
    },
    "data": {
        "jd": "SQL, Python, Excel, Tableau, Power BI, data cleaning, dashboards, ETL, statistics, stakeholder reporting.",
        "languages": ["Python", "SQL"],
        "aiml": ["Pandas"],
        "tools": ["Excel", "Tableau", "Power BI"],
        "bullets": [
            "Wrote SQL and Python (Pandas) for data cleaning and ETL",
            "Built Tableau / Power BI dashboards for weekly reporting",
            "Explained findings to stakeholders with Excel summaries",
        ],
    },
    "ml_ai": {
        "jd": "Python, PyTorch, TensorFlow, SQL, scikit-learn, model training, evaluation, LangChain, RAG, LLMs, Git.",
        "languages": ["Python", "SQL"],
        "aiml": ["PyTorch", "TensorFlow", "Scikit-learn", "LangChain", "RAG", "LLMs"],
        "tools": ["Git"],
        "bullets": [
            "Trained and evaluated models with PyTorch / TensorFlow / Scikit-learn",
            "Prototyped RAG flows with LangChain and LLMs",
            "Pulled labeled data with SQL and tracked experiments in Git",
        ],
    },
    "devops": {
        "jd": "Linux, Docker, Kubernetes, AWS, Terraform, CI/CD, Git, Python, monitoring, infrastructure as code.",
        "languages": ["Python", "Bash"],
        "tools": ["Git", "Docker", "Kubernetes", "AWS", "Terraform", "Linux", "CI/CD"],
        "bullets": [
            "Automated deploys with Docker, Kubernetes, and CI/CD",
            "Wrote Terraform for AWS lab environments",
            "Used Linux and Git for infrastructure as code reviews",
        ],
    },
    "security": {
        "jd": "SIEM, network security, Python, incident response, vulnerability scanning, IAM, Linux, Git, risk reporting.",
        "languages": ["Python"],
        "core_cs": ["Computer Networks"],
        "tools": ["Git", "Linux"],
        "bullets": [
            "Triaged SIEM alerts and documented incident response steps",
            "Wrote Python scripts to parse security logs on Linux",
            "Tracked vulnerabilities and IAM notes for weekly risk reporting",
        ],
    },
    "qa": {
        "jd": "test cases, Selenium, Java, API testing, Jira, Git, regression, automation, quality assurance, CI.",
        "languages": ["Java"],
        "tools": ["Git", "Jira", "Selenium"],
        "bullets": [
            "Wrote test cases and logged defects in Jira",
            "Automated regression with Selenium and Java",
            "Added API tests and ran suites in CI before release",
        ],
    },
    "mobile": {
        "jd": "Kotlin, Swift, Android, iOS, REST APIs, Git, mobile UI, Flutter, React Native.",
        "languages": ["Kotlin", "Swift"],
        "frontend": ["React Native"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "bullets": [
            "Built mobile screens in Kotlin / Swift / Flutter / React Native",
            "Fetched data from REST APIs",
            "Used Git for feature branches and reviews",
        ],
    },
    "business": {
        "jd": "requirements gathering, process mapping, Excel, SQL, user stories, stakeholders, Jira, documentation.",
        "languages": ["SQL"],
        "tools": ["Excel", "Jira"],
        "bullets": [
            "Gathered requirements and wrote user stories in Jira",
            "Mapped current-state processes for stakeholders",
            "Built Excel and SQL views for status reporting",
        ],
    },
    "finance": {
        "jd": "Excel, financial modeling, forecasting, variance analysis, reporting, GAAP, attention to detail.",
        "tools": ["Excel"],
        "bullets": [
            "Built Excel models for monthly forecasting",
            "Prepared variance analysis for leadership reviews",
            "Checked source figures before publishing reports",
        ],
    },
    "marketing": {
        "jd": "SEO, Google Analytics, content, campaigns, social media, keyword research, CMS, reporting.",
        "tools": ["Google Analytics", "Excel"],
        "bullets": [
            "Wrote SEO content and keyword research briefs",
            "Tracked campaigns in Google Analytics",
            "Scheduled social posts and reported results in Excel",
        ],
    },
    "sales": {
        "jd": "CRM, pipeline, outreach, qualification, negotiation, Excel, quota, customer communication.",
        "tools": ["Excel"],
        "bullets": [
            "Logged outreach and pipeline notes in the CRM",
            "Qualified inbound leads for account executives",
            "Tracked weekly activity in Excel",
        ],
    },
    "design": {
        "jd": "Figma, wireframes, prototyping, user research, accessibility, visual design, stakeholder feedback.",
        "tools": ["Figma"],
        "bullets": [
            "Created wireframes and prototypes in Figma",
            "Ran lightweight user research sessions",
            "Documented accessibility notes from stakeholder feedback",
        ],
    },
}

ROLE_FAMILY = {
    "Software Engineer": "software",
    "Software Developer": "software",
    "Software Development Engineer": "software",
    "SDE": "software",
    "Backend Engineer": "backend",
    "Backend Developer": "backend",
    "Frontend Engineer": "frontend",
    "Frontend Developer": "frontend",
    "Full Stack Engineer": "software",
    "Full Stack Developer": "software",
    "Web Developer": "frontend",
    "Java Developer": "backend",
    "Python Developer": "backend",
    "C++ Developer": "software",
    "JavaScript Developer": "frontend",
    "TypeScript Developer": "frontend",
    "React Developer": "frontend",
    "Node.js Developer": "backend",
    "Spring Boot Developer": "backend",
    "Django Developer": "backend",
    "Data Analyst": "data",
    "Data Scientist": "ml_ai",
    "Data Engineer": "data",
    "Business Intelligence Analyst": "data",
    "BI Developer": "data",
    "Analytics Engineer": "data",
    "SQL Developer": "data",
    "Reporting Analyst": "data",
    "Machine Learning Engineer": "ml_ai",
    "AI Engineer": "ml_ai",
    "ML Engineer": "ml_ai",
    "Deep Learning Engineer": "ml_ai",
    "NLP Engineer": "ml_ai",
    "Computer Vision Engineer": "ml_ai",
    "Generative AI Engineer": "ml_ai",
    "MLOps Engineer": "devops",
    "DevOps Engineer": "devops",
    "Cloud Engineer": "devops",
    "Cloud Architect": "devops",
    "Platform Engineer": "devops",
    "Site Reliability Engineer": "devops",
    "Infrastructure Engineer": "devops",
    "Cybersecurity Analyst": "security",
    "Security Engineer": "security",
    "SOC Analyst": "security",
    "Penetration Tester": "security",
    "QA Engineer": "qa",
    "SDET": "qa",
    "Automation Test Engineer": "qa",
    "Android Developer": "mobile",
    "iOS Developer": "mobile",
    "Flutter Developer": "mobile",
    "React Native Developer": "mobile",
    "Business Analyst": "business",
    "Product Manager": "business",
    "Project Coordinator": "business",
    "Financial Analyst": "finance",
    "Risk Analyst": "finance",
    "Digital Marketing Specialist": "marketing",
    "SEO Specialist": "marketing",
    "Content Marketing Specialist": "marketing",
    "Sales Executive": "sales",
    "Business Development Representative": "sales",
    "UI/UX Designer": "design",
    "Product Designer": "design",
    "Graphic Designer": "design",
    "Technical Writer": "business",
    "Technical Support Engineer": "business",
    "HR Coordinator": "business",
    "Operations Analyst": "data",
    "Supply Chain Analyst": "data",
    "Solutions Engineer": "software",
    "Integration Engineer": "backend",
    "Firmware Engineer": "software",
    "IoT Engineer": "software",
    "Network Engineer": "security",
    "Systems Administrator": "devops",
}


def _tech_block(bank):
    lines = []
    mapping = [
        ("Languages", "languages"),
        ("Core CS", "core_cs"),
        ("Frontend", "frontend"),
        ("Backend", "backend"),
        ("Databases", "databases"),
        ("AI/ML & GenAI", "aiml"),
        ("Tools & Cloud", "tools"),
    ]
    for label, key in mapping:
        vals = bank.get(key) or []
        if vals:
            lines.append(f"{label}: " + ", ".join(vals))
    return "\n".join(lines) if lines else "Tools & Cloud: Excel"


def _messy_skills(bank):
    bag = []
    for key in ("languages", "core_cs", "frontend", "backend", "databases", "aiml", "tools"):
        bag.extend(bank.get(key) or [])
    bag.extend(["communication", "teamwork", "python", "problem-solving"])
    return ", ".join(bag)


def _build_pair(role, company, bank, idx, variant):
    name, email = NAMES[idx % len(NAMES)]
    employer = EMPLOYERS[idx % len(EMPLOYERS)]
    title = role if variant == "experienced" else (role.replace("Engineer", "Intern").replace("Developer", "Intern").replace("Analyst", "Intern").replace("Specialist", "Intern").replace("Manager", "Intern").replace("Executive", "Intern").replace("Coordinator", "Intern").replace("Administrator", "Intern").replace("Tester", "Intern").replace("Designer", "Intern").replace("Writer", "Intern").replace("Representative", "Intern"))
    if title == role and variant != "experienced":
        title = role + " Intern"
    bullets = bank["bullets"]
    orig = f"""{name.title()}
{email}
Student targeting {role}. Skills: {_messy_skills(bank)}
EXPERIENCE
{title} | {employer} | 2024
- {bullets[0]}
- {bullets[1]}
- {bullets[2]}
EDUCATION
B.S. related field, 2025
"""
    assistant = f"""{name}
{email}

PROFESSIONAL SUMMARY
{title} at {employer} with hands-on work in {', '.join((bank.get('languages') or bank.get('tools') or ['core tools'])[:3])}. Comfortable with the keywords this {role} role requires, including Git-based collaboration where relevant. Seeking a {role} position at {company}.

TECHNICAL SKILLS
{_tech_block(bank)}

SOFT SKILLS
{SOFT_SKILLS_LINE}

PROFESSIONAL EXPERIENCE
{title} | {employer} | 2024
• {bullets[0]}
• {bullets[1]}
• {bullets[2]}

EDUCATION
B.S. related field, 2025
"""
    jd = bank["jd"]
    if variant == "keyword":
        jd = bank["jd"] + " Must-have keywords: " + ", ".join(
            (bank.get("languages") or []) + (bank.get("tools") or []) + (bank.get("aiml") or [])
        )
    return v6_example(
        f"v6_kw_{role.lower().replace(' ', '_').replace('/', '_')}_{variant}_{idx}",
        role,
        v6_user(company, role, jd, orig),
        assistant,
        category="role_keywords",
    )


COMPANY_BY_FAMILY = {
    "software": "Google",
    "frontend": "Adobe",
    "backend": "Amazon",
    "data": "Nielsen",
    "ml_ai": "NVIDIA",
    "devops": "IBM",
    "security": "Deloitte",
    "qa": "Microsoft",
    "mobile": "Samsung",
    "business": "McKinsey",
    "finance": "Goldman Sachs",
    "marketing": "HubSpot",
    "sales": "Oracle",
    "design": "Figma",
}

before = len(GOLD)
idx = 0
for role, family in ROLE_FAMILY.items():
    bank = FAMILY_BANKS[family]
    company = COMPANY_BY_FAMILY[family]
    GOLD.append(_build_pair(role, company, bank, idx, "intern"))
    idx += 1
    GOLD.append(_build_pair(role, company, bank, idx, "keyword"))
    idx += 1

# Extra keyword-placement drills: mixed dump -> labeled technical lines.
for family, bank in FAMILY_BANKS.items():
    mixed = _messy_skills(bank)
    GOLD.append(v6_example(
        f"v6_kw_split_{family}",
        family.replace("_", " ").title(),
        "Put every technical keyword on the correct TECHNICAL SKILLS line. Soft skills stay in SOFT SKILLS. Never put Python or Java under SOFT SKILLS.\n\nSKILLS: " + mixed,
        "TECHNICAL SKILLS\n" + _tech_block(bank) + "\n\nSOFT SKILLS\n" + SOFT_SKILLS_LINE + "\n",
        category="keyword_placement",
    ))

print("V6-03f added:", len(GOLD) - before)
print("gold examples total:", len(GOLD))
print("unique roles now:", len({row["role"] for row in GOLD}))


In [ ]:
# ============================================================
# V6-03g — MORE ROLES x MORE EXAMPLE VARIANTS
# Run after V6-03f. Adds intern / experienced / fresher / jumble
# rewrites for extra job families so V6-08 sees more diversity.
# ============================================================

if "SOFT_SKILLS_LINE" not in globals():
    SOFT_SKILLS_LINE = (
        "Communication, Collaboration, Problem Solving, Critical Thinking, "
        "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
    )

EXTRA_FAMILY_BANKS = {
    "software": {
        "jd": "Java, Python, SQL, Git, REST APIs, data structures, testing.",
        "languages": ["Java", "Python", "SQL"],
        "core_cs": ["Data Structures & Algorithms"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "project": "Campus API\n• Built a small REST API in Java/Python",
        "bullets": [
            "Built features with Java/Python and REST APIs",
            "Used Git and wrote tests",
            "Applied data structures in coding tasks",
        ],
    },
    "frontend": {
        "jd": "JavaScript, TypeScript, React, HTML5, CSS3, Git, REST APIs.",
        "languages": ["JavaScript", "TypeScript"],
        "frontend": ["React.js", "HTML5/CSS3"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "project": "Portfolio Site\n• Built a React.js UI with HTML5/CSS3",
        "bullets": [
            "Built React.js pages with HTML5/CSS3",
            "Consumed REST APIs",
            "Used Git for reviews",
        ],
    },
    "devops": {
        "jd": "Linux, Docker, AWS, CI/CD, Git, Python.",
        "languages": ["Python"],
        "tools": ["Git", "Docker", "AWS", "Linux", "CI/CD"],
        "project": "Lab Pipeline\n• Packaged an app with Docker and a CI job",
        "bullets": [
            "Packaged services with Docker",
            "Used AWS and Linux in a lab",
            "Kept changes in Git",
        ],
    },
    "qa": {
        "jd": "test cases, Selenium, Java, Jira, Git, regression.",
        "languages": ["Java"],
        "tools": ["Git", "Jira", "Selenium"],
        "project": "Regression Pack\n• Automated a small Selenium suite",
        "bullets": [
            "Wrote test cases in Jira",
            "Automated checks with Selenium and Java",
            "Logged defects before release",
        ],
    },
    "security": {
        "jd": "SIEM, Python, Linux, incident response, Git, network security.",
        "languages": ["Python"],
        "core_cs": ["Computer Networks"],
        "tools": ["Git", "Linux"],
        "project": "Log Parser\n• Wrote a Python script to parse security logs",
        "bullets": [
            "Triaged alerts and wrote incident notes",
            "Parsed logs with Python on Linux",
            "Tracked findings in Git",
        ],
    },
    "mobile": {
        "jd": "Kotlin, Swift, Android, iOS, REST APIs, Git.",
        "languages": ["Kotlin", "Swift"],
        "backend": ["REST APIs"],
        "tools": ["Git"],
        "project": "Campus App\n• Built a small mobile screen that calls a REST API",
        "bullets": [
            "Built mobile screens in Kotlin/Swift",
            "Fetched data from REST APIs",
            "Used Git for branches",
        ],
    },
    "embedded": {
        "jd": "C, C++, microcontrollers, firmware, RTOS, debugging, Git, hardware bring-up, oscilloscope.",
        "languages": ["C", "C++"],
        "tools": ["Git"],
        "project": "Sensor Node Firmware\n• Wrote C firmware for a microcontroller and debugged bring-up",
        "bullets": [
            "Wrote C/C++ firmware for a microcontroller lab",
            "Debugged hardware bring-up with basic instruments",
            "Versioned firmware in Git",
        ],
    },
    "networking": {
        "jd": "TCP/IP, routing, switching, Linux, firewall, DNS, DHCP, troubleshooting, documentation.",
        "core_cs": ["Computer Networks"],
        "tools": ["Linux"],
        "project": "Campus Lab Network\n• Documented a small routed lab with DNS and DHCP",
        "bullets": [
            "Configured lab routers and switches",
            "Troubleshot TCP/IP and DNS issues on Linux",
            "Documented firewall change notes",
        ],
    },
    "operations": {
        "jd": "process improvement, Excel, SQL, SOPs, stakeholder updates, KPI tracking, scheduling.",
        "languages": ["SQL"],
        "tools": ["Excel"],
        "project": "Shift KPI Tracker\n• Built an Excel tracker for weekly operations KPIs",
        "bullets": [
            "Tracked KPIs and schedules in Excel",
            "Wrote SQL to audit exceptions",
            "Updated SOPs after a small process change",
        ],
    },
    "customer": {
        "jd": "ticketing, customer communication, troubleshooting, CRM, SLAs, documentation, Excel.",
        "tools": ["Excel"],
        "project": "Help Center Drafts\n• Wrote short troubleshooting articles from ticket themes",
        "bullets": [
            "Resolved tickets within SLA using the help desk",
            "Logged customer notes in the CRM",
            "Documented repeat issues in Excel",
        ],
    },
    "legal": {
        "jd": "document review, calendaring, confidentiality, Excel, research, stakeholder coordination.",
        "tools": ["Excel"],
        "project": "Clause Tracker\n• Built an Excel tracker for contract dates and owners",
        "bullets": [
            "Tracked filing dates and calendars",
            "Prepared document sets for attorney review",
            "Kept confidential files organized",
        ],
    },
    "healthcare": {
        "jd": "scheduling, patient communication, HIPAA awareness, records, Excel, coordination.",
        "tools": ["Excel"],
        "project": "Clinic Schedule Sheet\n• Maintained an Excel sheet for clinic appointments",
        "bullets": [
            "Scheduled appointments and sent reminders",
            "Updated records with care-team notes",
            "Tracked supplies in Excel",
        ],
    },
    "education": {
        "jd": "program coordination, scheduling, communication, Excel, event logistics, documentation.",
        "tools": ["Excel"],
        "project": "Workshop Roster\n• Built an Excel roster for a student workshop series",
        "bullets": [
            "Coordinated workshop schedules and rooms",
            "Communicated updates to students and faculty",
            "Tracked attendance in Excel",
        ],
    },
    "hr": {
        "jd": "recruiting, onboarding, HRIS, scheduling, employee communication, Excel, ATS.",
        "tools": ["Excel"],
        "project": "Onboarding Checklist\n• Created an Excel onboarding checklist for new hires",
        "bullets": [
            "Scheduled interviews and sent candidate updates",
            "Prepared onboarding checklists",
            "Maintained records in the HRIS / Excel",
        ],
    },
    "product": {
        "jd": "roadmaps, user stories, Jira, analytics, stakeholder communication, prioritization, SQL.",
        "languages": ["SQL"],
        "tools": ["Jira", "Excel"],
        "project": "Feature Spec Mini\n• Wrote a one-page spec and user stories in Jira",
        "bullets": [
            "Wrote user stories and acceptance criteria in Jira",
            "Prioritized a small roadmap with stakeholders",
            "Reviewed funnel metrics with SQL and Excel",
        ],
    },
}

EXTRA_ROLES = {
    "Embedded Software Engineer": "embedded",
    "Firmware Engineer": "embedded",
    "IoT Engineer": "embedded",
    "Robotics Engineer": "embedded",
    "Hardware Engineer": "embedded",
    "Network Administrator": "networking",
    "Systems Administrator": "networking",
    "IT Support Engineer": "networking",
    "Systems Engineer": "networking",
    "Operations Coordinator": "operations",
    "Process Analyst": "operations",
    "Administrative Coordinator": "operations",
    "Customer Service Representative": "customer",
    "Technical Support Specialist": "customer",
    "Client Service Associate": "customer",
    "Legal Assistant": "legal",
    "Legal Operations Associate": "legal",
    "Healthcare Operations Associate": "healthcare",
    "Medical Administrative Assistant": "healthcare",
    "Education Coordinator": "education",
    "Academic Coordinator": "education",
    "Research Coordinator": "education",
    "Recruiter": "hr",
    "Talent Acquisition Coordinator": "hr",
    "HR Generalist": "hr",
    "Product Analyst": "product",
    "Technical Product Manager": "product",
    "Project Manager": "product",
    "Program Coordinator": "product",
    "Implementation Specialist": "product",
    "Technical Account Manager": "product",
    "Procurement Analyst": "operations",
    "Logistics Coordinator": "operations",
    "Inventory Analyst": "operations",
    "C Developer": "embedded",
    "TypeScript Developer": "frontend",
    "Release Engineer": "devops",
    "Build Engineer": "devops",
    "Performance Test Engineer": "qa",
    "Mobile Application Developer": "mobile",
    "Information Security Analyst": "security",
    "Cloud Security Engineer": "security",
    "Application Security Engineer": "security",
}

# Prefer 03f banks when present.
if "FAMILY_BANKS" in globals():
    for key, bank in EXTRA_FAMILY_BANKS.items():
        FAMILY_BANKS.setdefault(key, bank)
    BANKS = FAMILY_BANKS
else:
    BANKS = EXTRA_FAMILY_BANKS

NAMES_G = [
    ("RILEY ADAMS", "riley.adams@email.com"),
    ("ZARA KHAN", "zara.khan@email.com"),
    ("MATEO CRUZ", "mateo.cruz@email.com"),
    ("HANA SUZUKI", "hana.suzuki@email.com"),
    ("OWEN BLAKE", "owen.blake@email.com"),
    ("AMIRA SAID", "amira.said@email.com"),
    ("LEO FRANK", "leo.frank@email.com"),
    ("NIA BOATENG", "nia.boateng@email.com"),
]
LABS = ["Cedar Labs", "Pinewood Tech", "Harbor Apps", "Summit Grid", "Bluekite", "Fieldnote"]


def _tech_block_g(bank):
    lines = []
    for label, key in [
        ("Languages", "languages"),
        ("Core CS", "core_cs"),
        ("Frontend", "frontend"),
        ("Backend", "backend"),
        ("Databases", "databases"),
        ("AI/ML & GenAI", "aiml"),
        ("Tools & Cloud", "tools"),
    ]:
        vals = bank.get(key) or []
        if vals:
            lines.append(f"{label}: " + ", ".join(vals))
    return "\n".join(lines) if lines else "Tools & Cloud: Excel"


def _kw(bank):
    bag = []
    for key in ("languages", "core_cs", "frontend", "backend", "databases", "aiml", "tools"):
        bag.extend(bank.get(key) or [])
    return bag


def _assistant_full(name, email, summary, tech, exp_block, project, edu):
    proj = f"\n\nPROJECTS\n{project}" if project else ""
    return f"""{name}
{email}

PROFESSIONAL SUMMARY
{summary}

TECHNICAL SKILLS
{tech}

SOFT SKILLS
{SOFT_SKILLS_LINE}
{exp_block}{proj}

EDUCATION
{edu}
"""


def _add(uid, role, user, assistant):
    GOLD.append(v6_example(uid, role, user, assistant, category="role_examples_v2"))


before = len(GOLD)
i = 0
roles = dict(EXTRA_ROLES)
if "ROLE_FAMILY" in globals():
    for role, family in ROLE_FAMILY.items():
        roles.setdefault(role, family)

for role, family in roles.items():
    bank = BANKS.get(family) or EXTRA_FAMILY_BANKS.get(family) or EXTRA_FAMILY_BANKS["operations"]
    name, email = NAMES_G[i % len(NAMES_G)]
    lab = LABS[i % len(LABS)]
    kws = _kw(bank)
    tech = _tech_block_g(bank)
    jd = bank.get("jd") or ", ".join(kws)
    bullets = bank.get("bullets") or [
        f"Used {kws[0]} on day-to-day work" if kws else "Completed assigned team work",
        "Documented progress for the team",
        "Collaborated using shared tools",
    ]
    project = bank.get("project", "")
    edu = "B.S. related field, 2025"
    i += 1

    # 1) Intern with real employer
    intern_title = role if "Intern" in role else f"{role} Intern"
    orig_intern = f"""{name.title()}
{email}
Skills: {', '.join(kws + ['communication', 'python', 'teamwork'])}
EXPERIENCE
{intern_title} | {lab} | 2024
- {bullets[0]}
- {bullets[1]}
EDUCATION
{edu}
"""
    _add(
        f"v6_g_intern_{i}",
        role,
        v6_user("TargetCo", role, jd, orig_intern),
        _assistant_full(
            name, email,
            f"{intern_title} at {lab}. Uses {', '.join(kws[:3]) or 'core tools'} in daily work. Seeking a {role} role at TargetCo.",
            tech,
            f"\nPROFESSIONAL EXPERIENCE\n{intern_title} | {lab} | 2024\n• {bullets[0]}\n• {bullets[1]}\n• {bullets[2]}",
            "",
            edu,
        ),
    )

    # 2) Experienced — two real jobs, never invent TargetCo
    orig_exp = f"""{name.title()}
{email}
EXPERIENCE
{role} | {lab} | 2023 - Present
- {bullets[0]}
- {bullets[1]}
Junior {role} | {LABS[(i+1) % len(LABS)]} | 2021 - 2023
- {bullets[2]}
SKILLS: {', '.join(kws)}
EDUCATION
{edu}
"""
    _add(
        f"v6_g_exp_{i}",
        role,
        v6_user("TargetCo", role, jd, orig_exp),
        _assistant_full(
            name, email,
            f"{role} at {lab} with earlier work at {LABS[(i+1) % len(LABS)]}. Strengths include {', '.join(kws[:4]) or 'core tools'}. Interested in the {role} role at TargetCo.",
            tech,
            f"\nPROFESSIONAL EXPERIENCE\n{role} | {lab} | 2023 – Present\n• {bullets[0]}\n• {bullets[1]}\n\nJunior {role} | {LABS[(i+1) % len(LABS)]} | 2021 – 2023\n• {bullets[2]}",
            "",
            edu,
        ),
    )

    # 3) Fresher — projects only, no fake job
    orig_fr = f"""{name.title()}
{email}
CS/related student. No internship.
PROJECTS
{project or (role + ' Capstone') }
Skills: {', '.join(kws + ['communication'])}
EDUCATION
{edu}
"""
    _add(
        f"v6_g_fresher_{i}",
        role,
        v6_user("TargetCo", role, jd, orig_fr),
        _assistant_full(
            name, email,
            f"Graduate focused on {role.lower()} skills including {', '.join(kws[:3]) or 'core tools'}. Academic project work only. Seeking a {role} role at TargetCo.",
            tech,
            "",
            project or f"{role} Capstone\n• Applied {', '.join(kws[:3]) or 'course tools'} in an academic project",
            edu,
        ),
    )

    # 4) Jumble rewrite — keywords dumped into one line
    jumble = f"""resume
PROFESSIONAL SUMMARY
{role.lower()} targeting targetco, skilled in {', '.join(kws)}, communication, python, teamwork. I am e
TECHNICAL SKILLS
Languages: English, {', '.join(kws)}, problem-solving, software, engineer
"""
    _add(
        f"v6_g_jumble_{i}",
        role,
        "Rewrite this jumbled resume. Keep facts. Separate technical lines. Soft skills only in SOFT SKILLS. Do not invent jobs.\n\n" + jumble,
        _assistant_full(
            name, email,
            f"Candidate targeting a {role} role with skills in {', '.join(kws[:4]) or 'core tools'}.",
            tech,
            "",
            "",
            edu,
        ),
    )

print("V6-03g added:", len(GOLD) - before)
print("gold examples total:", len(GOLD))
print("unique roles:", len({row.get('role') for row in GOLD}))


In [ ]:
# ============================================================
# V6-03h — JD-ALIGNED SUMMARIES + MORE ROLES
# Run after V6-03g and before V6-04.
# Teaches a 3-sentence summary: who they are, JD skills they
# already have, and real project names. Never invents employers.
# ============================================================

if "GOLD" not in globals() or "v6_example" not in globals() or "v6_user" not in globals():
    raise RuntimeError("Run V6-03 through V6-03g before V6-03h.")

if "SOFT_SKILLS_LINE" not in globals():
    SOFT_SKILLS_LINE = (
        "Communication, Collaboration, Problem Solving, Critical Thinking, "
        "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
    )


def _summary_resume(name, email, summary, skills, experience, projects, education):
    exp = f"\n\nPROFESSIONAL EXPERIENCE\n{experience}" if experience else ""
    proj = f"\n\nPROJECTS\n{projects}" if projects else ""
    return (
        f"{name}\n{email}\n\n"
        f"PROFESSIONAL SUMMARY\n{summary}\n\n"
        f"TECHNICAL SKILLS\n{skills}\n\n"
        f"SOFT SKILLS\n{SOFT_SKILLS_LINE}"
        f"{exp}{proj}\n\nEDUCATION\n{education}\n"
    )


SUMMARY_CASES = [
    {
        "id": "v6_h_sum_backend",
        "role": "Backend Engineer",
        "company": "Stripe",
        "jd": "Java, Spring Boot, FastAPI, PostgreSQL, Docker, AWS. React is a plus.",
        "name": "SURAKANTI SAHITHA",
        "email": "sahithareddy06@gmail.com",
        "orig_summary": "Final-year Computer Science undergraduate with expertise in Java, Spring Boot, React.js, and FastAPI.",
        "skills": "Languages: Java, SQL\nBackend: Spring Boot, FastAPI, REST APIs\nDatabases: PostgreSQL\nTools & Cloud: Git, Docker",
        "projects": "AeroTrack AI\nReact.js, TypeScript, FastAPI\n\u2022 Built a flight-tracking API\n\nAI-Powered Smart Farming Platform\nReact.js, Spring Boot\n\u2022 Built a farming web app",
        "education": "B.Tech in Computer Science, Anurag University, 2023 \u2013 2027",
        "summary": (
            "Final-year Computer Science undergraduate targeting the Backend Engineer role at Stripe. "
            "Hands-on experience with Java, Spring Boot, FastAPI, and React.js matches the skills this role emphasizes. "
            "That experience is demonstrated in AeroTrack AI and AI-Powered Smart Farming Platform."
        ),
    },
    {
        "id": "v6_h_sum_data",
        "role": "Data Analyst",
        "company": "Deloitte",
        "jd": "SQL, Python, Excel, Tableau, dashboards, stakeholder reporting.",
        "name": "MEERA JOSHI",
        "email": "meera.joshi@email.com",
        "orig_summary": "Statistics student who analyzes survey data with Python and SQL.",
        "skills": "Languages: Python, SQL\nTools & Cloud: Excel, Tableau",
        "projects": "Campus Survey Dashboard\n\u2022 Cleaned survey rows in Python and charted results in Tableau",
        "education": "B.S. Statistics, 2025",
        "summary": (
            "Statistics student targeting the Data Analyst role at Deloitte. "
            "Hands-on experience with SQL, Python, Excel, and Tableau matches the skills this role emphasizes. "
            "That experience is demonstrated in Campus Survey Dashboard."
        ),
    },
    {
        "id": "v6_h_sum_ml",
        "role": "Machine Learning Engineer",
        "company": "NVIDIA",
        "jd": "Python, PyTorch, SQL, model evaluation, Git. Kubernetes is a plus.",
        "name": "ADITYA RAO",
        "email": "aditya.rao@email.com",
        "orig_summary": "Computer science student building classifiers with PyTorch.",
        "skills": "Languages: Python, SQL\nAI/ML & GenAI: PyTorch, Scikit-learn\nTools & Cloud: Git",
        "projects": "Image Classifier Lab\n\u2022 Trained and evaluated a PyTorch classifier on a class dataset",
        "education": "B.Tech Computer Science, 2025",
        "summary": (
            "Computer science student targeting the Machine Learning Engineer role at NVIDIA. "
            "Hands-on experience with Python, PyTorch, SQL, and Git matches the skills this role emphasizes. "
            "That experience is demonstrated in Image Classifier Lab."
        ),
    },
    {
        "id": "v6_h_sum_frontend",
        "role": "Frontend Developer",
        "company": "Adobe",
        "jd": "JavaScript, React, HTML, CSS, accessibility, Git.",
        "name": "SANA KHAN",
        "email": "sana.khan@email.com",
        "orig_summary": "Frontend intern who builds React pages.",
        "skills": "Languages: JavaScript\nFrontend: React.js, HTML5/CSS3\nTools & Cloud: Git",
        "experience": "Frontend Intern | PixelWorks | Jan 2024 \u2013 Jun 2024\n\u2022 Built React pages for an internal dashboard\n\u2022 Styled layouts with HTML and CSS",
        "projects": "",
        "education": "B.S. Computer Science, 2025",
        "summary": (
            "Frontend Intern at PixelWorks targeting the Frontend Developer role at Adobe. "
            "Hands-on experience with JavaScript, React.js, HTML, CSS, and Git matches the skills this role emphasizes."
        ),
    },
    {
        "id": "v6_h_sum_security",
        "role": "Cybersecurity Analyst",
        "company": "Deloitte",
        "jd": "SIEM, Python, incident response, Linux, network security.",
        "name": "KAREN LOPEZ",
        "email": "karen.lopez@email.com",
        "orig_summary": "Security intern who triages lab alerts.",
        "skills": "Languages: Python\nCore CS: Computer Networks\nTools & Cloud: Linux, Git",
        "experience": "Security Intern | ShieldByte | 2024\n\u2022 Triaged SIEM alerts in a lab SOC\n\u2022 Parsed logs with Python on Linux",
        "projects": "",
        "education": "B.S. Cybersecurity, 2025",
        "summary": (
            "Security Intern at ShieldByte targeting the Cybersecurity Analyst role at Deloitte. "
            "Hands-on experience with Python, Linux, and SIEM alert triage matches the skills this role emphasizes."
        ),
    },
]

before = len(GOLD)
for case in SUMMARY_CASES:
    original = (
        f"{case['name'].title()}\n{case['email']}\n"
        f"SUMMARY\n{case['orig_summary']}\n"
        + (f"EXPERIENCE\n{case['experience']}\n" if case.get('experience') else "")
        + (f"PROJECTS\n{case['projects']}\n" if case.get('projects') else "")
        + f"EDUCATION\n{case['education']}\n"
    )
    assistant = _summary_resume(
        case["name"],
        case["email"],
        case["summary"],
        case["skills"],
        case.get("experience", ""),
        case.get("projects", ""),
        case["education"],
    )
    GOLD.append(v6_example(
        case["id"],
        case["role"],
        v6_user(case["company"], case["role"], case["jd"], original),
        assistant,
        category="jd_summary",
    ))

# Extra roles not covered by V6-03e/03g. Two variants each: intern and fresher.
NEW_ROLE_BANKS = [
    ("Data Engineer", "Snowflake", "Python, SQL, ETL, Airflow, data pipelines, Git.",
     ["Python", "SQL"], ["Git"], "Sales ETL Lab", "Built a Python ETL job that loaded CSV extracts into SQL tables"),
    ("Machine Learning Engineer", "OpenAI", "Python, PyTorch, model evaluation, SQL, Git.",
     ["Python", "SQL"], ["Git", "PyTorch"], "Baseline Ranker", "Trained and evaluated a PyTorch ranking baseline"),
    ("Business Analyst", "McKinsey", "Excel, SQL, requirements, stakeholder reporting, process mapping.",
     ["SQL"], ["Excel"], "Process Map", "Mapped a campus service process and summarized volumes in Excel"),
    ("UX Designer", "Figma", "Figma, wireframes, prototyping, user research, accessibility.",
     [], ["Figma"], "Library App Wireframes", "Prototyped a library app flow in Figma from five student interviews"),
    ("Technical Writer", "Atlassian", "documentation, API docs, Markdown, clarity, developer communication.",
     [], ["Git"], "API Guide", "Wrote a Markdown guide for a class REST API"),
    ("Financial Analyst", "Goldman Sachs", "Excel, forecasting, variance analysis, financial modeling, reporting.",
     [], ["Excel"], "Budget Variance Sheet", "Built an Excel model comparing budget and actuals for a student club"),
    ("Platform Engineer", "Shopify", "Linux, Docker, CI/CD, Python, Git, AWS.",
     ["Python"], ["Linux", "Docker", "Git", "AWS"], "Deploy Lab", "Packaged a class service with Docker and a small CI script"),
    ("GenAI Engineer", "Cohere", "Python, LangChain, RAG, LLMs, FastAPI, Git.",
     ["Python"], ["Git", "FastAPI", "LangChain"], "Notes RAG Bot", "Built a small RAG bot over class notes with LangChain and FastAPI"),
    ("Power BI Developer", "Microsoft", "Power BI, SQL, Excel, dashboards, data modeling.",
     ["SQL"], ["Power BI", "Excel"], "Enrollment Dashboard", "Modeled enrollment tables in SQL and published a Power BI report"),
    ("Solutions Engineer", "Twilio", "REST APIs, Python, demos, customer communication, Git.",
     ["Python"], ["Git", "REST APIs"], "SMS Demo", "Demoed a Python script that called a messaging REST API"),
]

for idx, (role, company, jd, languages, tools, project, bullet) in enumerate(NEW_ROLE_BANKS, start=1):
    name = ["PRIYA MENON", "LUKAS BERG", "AISHA DIALLO", "TOMAS REYES", "YUNA CHO"][idx % 5]
    email = name.lower().replace(" ", ".") + "@email.com"
    skill_lines = []
    if languages:
        skill_lines.append("Languages: " + ", ".join(languages))
    if tools:
        skill_lines.append("Tools & Cloud: " + ", ".join(tools))
    skills = "\n".join(skill_lines) or "Tools & Cloud: Excel"
    shown = ", ".join((languages + tools)[:4]) or "core tools"
    edu = "B.S. related field, 2025"
    # Fresher: projects only. Do not invent an employer.
    original = (
        f"{name.title()}\n{email}\nNo internship.\n"
        f"PROJECTS\n{project}\n- {bullet}\n"
        f"SKILLS: {shown}\nEDUCATION\n{edu}\n"
    )
    summary = (
        f"Student targeting the {role} role at {company}. "
        f"Hands-on experience with {shown} matches the skills this role emphasizes. "
        f"That experience is demonstrated in {project}."
    )
    GOLD.append(v6_example(
        f"v6_h_role_{idx}",
        role,
        v6_user(company, role, jd, original),
        _summary_resume(name, email, summary, skills, "", f"{project}\n\u2022 {bullet}", edu),
        category="jd_summary_roles",
    ))

print("V6-03h added:", len(GOLD) - before)
print("gold examples total:", len(GOLD))
print("unique roles:", len({row.get('role') for row in GOLD}))


In [ ]:
# ============================================================
# V6-03i — FULL-PAGE RESUMES (do not stop after skills)
# Run after V6-03h, before V6-04.
# The broken output dropped projects, internships, and education.
# These examples always keep those sections.
# ============================================================

if "GOLD" not in globals() or "v6_example" not in globals() or "v6_user" not in globals():
    raise RuntimeError("Run V6-03 through V6-03h before V6-03i.")

FULL_PAGE = '''ANANYA RAO
ananya.rao@email.com

PROFESSIONAL SUMMARY
Final-year Computer Science undergraduate targeting the Software Engineer role at Microsoft. Hands-on experience with Java, React.js, and SQL matches the skills this role emphasizes. That experience is demonstrated in Campus Portal and Notes RAG Bot.

TECHNICAL SKILLS
Languages: Java, SQL
Frontend: React.js
Backend: Spring Boot, REST APIs
Tools & Cloud: Git

SOFT SKILLS
Communication, Collaboration, Problem Solving, Critical Thinking, Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership

PROJECTS
Campus Portal
React.js, Spring Boot, SQL
• Built a course portal with React.js and Spring Boot
• Stored records in SQL

Notes RAG Bot
• Prototyped a notes assistant with course documents

INTERNSHIP EXPERIENCE
Software Engineer Intern | Northwind Apps | Jun 2024 – Aug 2024
• Added REST API endpoints for the student portal
• Fixed defects found in testing

EDUCATION
B.Tech in Computer Science, 2023 – 2027

CERTIFICATIONS
• AWS Cloud Practitioner
'''

ORIGINAL = '''Ananya Rao
ananya.rao@email.com
Summary
Final-year Computer Science undergraduate.
Projects
Campus Portal
- Built a course portal with React and Spring Boot
Notes RAG Bot
- Prototyped a notes assistant
Internship Experience
Software Engineer Intern | Northwind Apps | Jun 2024 - Aug 2024
- Added REST endpoints
Education
B.Tech in Computer Science, 2023 - 2027
Certifications
- AWS Cloud Practitioner
'''

before = len(GOLD)
GOLD.append(v6_example(
    "v6_i_full_page",
    "Software Engineer",
    v6_user(
        "Microsoft",
        "Software Engineer",
        "Java, React, SQL, Spring Boot, REST APIs, Git.",
        ORIGINAL,
    ),
    FULL_PAGE,
    category="full_page_keep_sections",
))

# Same rule for a fresher: projects and education stay, no invented job.
FRESHER = '''RIYA SEN
riya.sen@email.com

PROFESSIONAL SUMMARY
Computer Science student targeting the Data Analyst role at Deloitte. Hands-on experience with SQL, Python, and Excel matches the skills this role emphasizes. That experience is demonstrated in Survey Dashboard.

TECHNICAL SKILLS
Languages: Python, SQL
Tools & Cloud: Excel

SOFT SKILLS
Communication, Collaboration, Problem Solving, Critical Thinking, Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership

PROJECTS
Survey Dashboard
• Cleaned survey rows with Python and summarized them in Excel

EDUCATION
B.S. Statistics, 2025
'''
GOLD.append(v6_example(
    "v6_i_fresher_keeps_projects",
    "Data Analyst",
    v6_user(
        "Deloitte",
        "Data Analyst",
        "SQL, Python, Excel, dashboards.",
        "Riya Sen\nriya.sen@email.com\nNo internship.\nPROJECTS\nSurvey Dashboard\n- Cleaned survey rows with Python\nEDUCATION\nB.S. Statistics, 2025\n",
    ),
    FRESHER,
    category="full_page_keep_sections",
))

print("V6-03i added:", len(GOLD) - before)
print("gold examples total:", len(GOLD))
for row in GOLD[-2:]:
    text = row["messages"][-1]["content"]
    for header in ("PROJECTS", "EDUCATION", "PROFESSIONAL SUMMARY"):
        print(row["id"], header, header in text)


In [ ]:
# ============================================================
# V6-03j — MORE TECH AND NON-TECH ROLES
# Run after V6-03i and before V6-04.
# Fresher examples only: real projects, no invented employers.
# Tech roles keep tools on technical lines. Non-tech roles do not
# gain programming languages the resume never had.
# ============================================================

if "GOLD" not in globals() or "v6_example" not in globals() or "v6_user" not in globals():
    raise RuntimeError("Run V6-03 through V6-03i before V6-03j.")

if "SOFT_SKILLS_LINE" not in globals():
    SOFT_SKILLS_LINE = (
        "Communication, Collaboration, Problem Solving, Critical Thinking, "
        "Adaptability, Time Management, Attention to Detail, Leadership, Initiative, Ownership"
    )


def _page(name, email, summary, skills, project, bullet, education):
    return (
        f"{name}\n{email}\n\n"
        f"PROFESSIONAL SUMMARY\n{summary}\n\n"
        f"TECHNICAL SKILLS\n{skills}\n\n"
        f"SOFT SKILLS\n{SOFT_SKILLS_LINE}\n\n"
        f"PROJECTS\n{project}\n\u2022 {bullet}\n\n"
        f"EDUCATION\n{education}\n"
    )


# kind, role, company, jd, skill line, project, bullet, education
MORE_ROLES = [
    # --- tech ---
    ("tech", ".NET Developer", "Microsoft", "C#, .NET, SQL, REST APIs, Git.",
     "Languages: C#\nBackend: .NET, REST APIs\nDatabases: SQL\nTools & Cloud: Git",
     "Help Desk Ticket API", "Built a C# .NET API that stored tickets in SQL", "B.S. Computer Science, 2025"),
    ("tech", "Golang Developer", "Cloudflare", "Go, REST APIs, SQL, Git, Linux.",
     "Languages: Go, SQL\nBackend: REST APIs\nTools & Cloud: Git, Linux",
     "Status Service", "Wrote a small Go service that returned health checks", "B.S. Computer Science, 2025"),
    ("tech", "Game Developer", "Unity", "C#, Unity, Git, gameplay programming.",
     "Languages: C#\nTools & Cloud: Git, Unity",
     "Campus Runner", "Built a 2D Unity game and tracked changes in Git", "B.S. Computer Science, 2025"),
    ("tech", "Database Administrator", "Oracle", "SQL, PostgreSQL, backups, indexing, access control.",
     "Languages: SQL\nDatabases: PostgreSQL\nTools & Cloud: Git",
     "Course DB Backup", "Wrote backup and index scripts for a class PostgreSQL database", "B.S. Information Systems, 2025"),
    ("tech", "Salesforce Developer", "Salesforce", "Apex, SOQL, Lightning, Git, CRM.",
     "Tools & Cloud: Salesforce, Git",
     "Club CRM Page", "Built a Lightning page and SOQL query for a student club CRM lab", "B.S. Information Systems, 2025"),
    ("tech", "Scrum Master", "Atlassian", "Scrum, Jira, standups, sprint planning, stakeholder communication.",
     "Tools & Cloud: Jira",
     "Sprint Board", "Ran a class sprint in Jira and wrote the retrospective notes", "B.S. Information Systems, 2025"),
    ("tech", "Kubernetes Engineer", "Red Hat", "Kubernetes, Docker, Linux, CI/CD, Git.",
     "Tools & Cloud: Kubernetes, Docker, Linux, Git",
     "Deploy Lab", "Packaged a class app with Docker and deployed it to a local Kubernetes cluster", "B.S. Computer Science, 2025"),
    ("tech", "Prompt Engineer", "Anthropic", "prompt design, LLMs, evaluation, Python, documentation.",
     "Languages: Python\nAI/ML & GenAI: LLMs\nTools & Cloud: Git",
     "Prompt Eval Sheet", "Compared prompt variants and recorded which answers stayed factual", "B.S. Computer Science, 2025"),
    ("tech", "ETL Developer", "Informatica", "SQL, Python, ETL, data pipelines, Git.",
     "Languages: Python, SQL\nTools & Cloud: Git",
     "Sales CSV Loader", "Loaded CSV extracts into SQL tables with a Python ETL script", "B.S. Information Systems, 2025"),
    ("tech", "Help Desk Technician", "ServiceNow", "ticketing, Windows, troubleshooting, customer communication.",
     "Tools & Cloud: Windows, ServiceNow",
     "Lab Ticket Queue", "Triaged computer-lab tickets and documented the fix steps", "A.S. Information Technology, 2025"),
    ("tech", "Desktop Support Technician", "Dell", "Windows, hardware troubleshooting, imaging, documentation.",
     "Tools & Cloud: Windows",
     "Lab Image Checklist", "Imaged lab PCs and wrote a checklist for the next setup", "A.S. Information Technology, 2025"),
    ("tech", "API Developer", "Postman", "REST APIs, JSON, Python, Git, testing.",
     "Languages: Python\nBackend: REST APIs\nTools & Cloud: Git, Postman",
     "Catalog API", "Built a REST API that returned catalog JSON and checked it in Postman", "B.S. Computer Science, 2025"),
    # --- non-tech ---
    ("nontech", "Accountant", "Deloitte", "Excel, reconciliations, journal entries, month-end close, attention to detail.",
     "Tools & Cloud: Excel",
     "Club Ledger", "Reconciled a student-club ledger in Excel against bank totals", "B.S. Accounting, 2025"),
    ("nontech", "Bookkeeper", "Intuit", "Excel, accounts payable, accounts receivable, invoices, recordkeeping.",
     "Tools & Cloud: Excel",
     "Invoice Log", "Tracked invoices and payments for a campus fundraiser in Excel", "A.S. Accounting, 2025"),
    ("nontech", "Auditor", "PwC", "Excel, sampling, workpapers, documentation, internal controls.",
     "Tools & Cloud: Excel",
     "Sample Workpaper", "Built an Excel sample of club expenses and noted missing receipts", "B.S. Accounting, 2025"),
    ("nontech", "Content Writer", "HubSpot", "writing, editing, SEO, content calendar, research.",
     "Tools & Cloud: Google Docs",
     "Campus Blog Series", "Wrote and edited four campus blog posts and tracked them on a content calendar", "B.A. English, 2025"),
    ("nontech", "Social Media Manager", "Nike", "content calendar, Instagram, analytics, captions, community replies.",
     "Tools & Cloud: Instagram, Excel",
     "Club Instagram Plan", "Scheduled a month of club posts and logged reach in Excel", "B.A. Communications, 2025"),
    ("nontech", "Copywriter", "Ogilvy", "headlines, editing, brand voice, research, deadlines.",
     "Tools & Cloud: Google Docs",
     "Fundraiser Headlines", "Wrote headline options and a short blurb for a campus fundraiser", "B.A. English, 2025"),
    ("nontech", "Event Coordinator", "Marriott", "scheduling, vendor notes, run-of-show, Excel, guest communication.",
     "Tools & Cloud: Excel",
     "Career Fair Run of Show", "Built the run-of-show and volunteer schedule for a campus career fair", "B.A. Communications, 2025"),
    ("nontech", "Executive Assistant", "Accenture", "calendar management, travel notes, meeting minutes, Excel, confidentiality.",
     "Tools & Cloud: Excel, Google Calendar",
     "Faculty Meeting Notes", "Kept a faculty meeting calendar and wrote the action-item notes", "B.A. Business, 2025"),
    ("nontech", "Office Manager", "WeWork", "scheduling, supplies, vendor follow-up, Excel, office procedures.",
     "Tools & Cloud: Excel",
     "Supply Tracker", "Tracked office-supply orders and costs for a student organization", "B.A. Business, 2025"),
    ("nontech", "Librarian", "Public Library", "cataloging, reference help, research, circulation, Excel.",
     "Tools & Cloud: Excel",
     "Reading List Catalog", "Cataloged a class reading list and helped classmates find sources", "B.A. English, 2025"),
    ("nontech", "Teacher", "Public School", "lesson plans, classroom management, assessment, communication, Excel.",
     "Tools & Cloud: Excel, Google Docs",
     "Algebra Lesson Set", "Wrote three lesson plans and a short quiz for a tutoring group", "B.Ed., 2025"),
    ("nontech", "Public Relations Specialist", "Edelman", "press notes, media lists, writing, stakeholder updates, Excel.",
     "Tools & Cloud: Excel, Google Docs",
     "Club Press Note", "Drafted a press note and media list for a campus event", "B.A. Communications, 2025"),
    ("nontech", "Retail Supervisor", "Target", "scheduling, customer service, inventory counts, cash handling, coaching.",
     "Tools & Cloud: Excel",
     "Shift Schedule", "Built a weekend shift schedule and an inventory count sheet", "B.A. Business, 2025"),
    ("nontech", "Customer Success Manager", "Zendesk", "onboarding notes, customer communication, renewal tracking, Excel.",
     "Tools & Cloud: Excel",
     "Onboarding Checklist", "Wrote an onboarding checklist and tracked follow-ups for a class client project", "B.A. Business, 2025"),
    ("nontech", "Grant Writer", "United Way", "proposal writing, budgets, research, deadlines, Excel.",
     "Tools & Cloud: Excel, Google Docs",
     "Club Grant Draft", "Drafted a two-page grant narrative and a simple Excel budget", "B.A. English, 2025"),
    ("nontech", "Paralegal", "Law Firm", "document review, filing, legal research, calendaring, writing.",
     "Tools & Cloud: Excel",
     "Case Chronology", "Built a chronology and filing checklist for a mock case file", "B.A. Political Science, 2025"),
    ("nontech", "Hotel Front Desk Associate", "Hilton", "reservations, guest communication, check-in, Excel, problem solving.",
     "Tools & Cloud: Excel",
     "Reservation Log", "Kept a reservation log and arrival notes for a student-run event hotel block", "A.S. Hospitality, 2025"),
    ("nontech", "Video Editor", "YouTube Studio", "Premiere Pro, cutting, captions, storytelling, deadlines.",
     "Tools & Cloud: Premiere Pro",
     "Club Recap Video", "Edited a three-minute club recap and added captions", "B.A. Media Studies, 2025"),
]

NAMES_J = [
    "ANIKA SHARMA", "DIEGO MARTINEZ", "FATIMA NOOR", "LIAM OCONNOR", "SOFIA ROSSI",
    "HARUTO SATO", "AMARA NWOSU", "JONAH LEVI",
]
before = len(GOLD)
seen_roles = {str(row.get("role", "")).lower() for row in GOLD}
added_tech = 0
added_nontech = 0
for idx, (kind, role, company, jd, skills, project, bullet, education) in enumerate(MORE_ROLES):
    if role.lower() in seen_roles:
        continue
    name = NAMES_J[idx % len(NAMES_J)]
    email = name.lower().replace(" ", ".") + "@email.com"
    shown = skills.replace("\n", ", ")
    original = (
        f"{name.title()}\n{email}\nNo full-time job.\n"
        f"PROJECTS\n{project}\n- {bullet}\n"
        f"SKILLS: {shown}\nEDUCATION\n{education}\n"
    )
    summary = (
        f"Student targeting the {role} role at {company}. "
        f"Project work with the tools this role uses is demonstrated in {project}."
    )
    GOLD.append(v6_example(
        f"v6_j_{kind}_{idx}",
        role,
        v6_user(company, role, jd, original),
        _page(name, email, summary, skills, project, bullet, education),
        category="tech_and_nontech_roles",
    ))
    seen_roles.add(role.lower())
    if kind == "tech":
        added_tech += 1
    else:
        added_nontech += 1

print("V6-03j added:", len(GOLD) - before)
print("new tech roles:", added_tech, "new non-tech roles:", added_nontech)
print("gold examples total:", len(GOLD))
print("unique roles:", len(seen_roles))


In [ ]:
# ============================================================
# V6-04 — MIX GOLD + FILTERED PRIOR DATA (drop jumbled assistants)
# Looks up the real V4/V5 jsonl paths used earlier in this notebook.
# ============================================================

def load_jsonl_if_exists(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    print(f"  loaded {len(rows):5d}  {path}")
    return rows


search_files = [
    PROJECT_ROOT / "datasets" / "v5" / "train_v5.jsonl",
    PROJECT_ROOT / "datasets" / "v5" / "extra_safety_v5.jsonl",
    PROJECT_ROOT / "datasets" / "v4" / "training" / "train_v4.jsonl",
    PROJECT_ROOT / "datasets" / "v4" / "training" / "master_v4.jsonl",
    PROJECT_ROOT / "datasets" / "v3" / "train_v3.jsonl",
]
# Also pick up any other train_*.jsonl under datasets, without duplicating.
seen_paths = {p.resolve() for p in search_files if p.exists()}
datasets_root = PROJECT_ROOT / "datasets"
if datasets_root.exists():
    for path in datasets_root.rglob("*.jsonl"):
        name = path.name.lower()
        if "train" in name or name.startswith("master"):
            if path.resolve() not in seen_paths:
                search_files.append(path)
                seen_paths.add(path.resolve())

print("Scanning prior jsonl files:")
prior = []
for path in search_files:
    prior.extend(load_jsonl_if_exists(path))

kept_prior = []
dropped = Counter()
for row in prior:
    messages = row.get("messages") or []
    if not isinstance(messages, list) or len(messages) < 2:
        dropped["structure"] += 1
        continue
    if str(messages[-1].get("role", "")).lower() != "assistant":
        dropped["no_assistant"] += 1
        continue
    assistant = str(messages[-1].get("content", "")).strip()
    if len(assistant) < 80:
        dropped["short"] += 1
        continue
    if is_jumbled_resume(assistant):
        dropped["jumble"] += 1
        continue
    if not has_professional_structure(assistant) and len(assistant) < 300:
        dropped["weak_format"] += 1
        continue
    kept_prior.append({"messages": messages})

# Oversample gold so format dominates even if prior is large.
gold_msgs = [{"messages": row["messages"]} for row in GOLD]
repeat = 8 if len(kept_prior) > 200 else 6
gold_boost = gold_msgs * repeat
if kept_prior:
    cap = max(len(gold_boost), int(len(gold_boost) * 1.5))
    rng_prior = random.Random(42)
    rng_prior.shuffle(kept_prior)
    kept_prior = kept_prior[:cap]

v6_rows = gold_boost + kept_prior
random.seed(42)
random.shuffle(v6_rows)

print("gold unique:", len(GOLD), "gold after oversample:", len(gold_boost))
print("prior kept:", len(kept_prior), "dropped:", dict(dropped), "v6 mix:", len(v6_rows))
if len(v6_rows) < 20:
    print("WARNING: very small mix; gold examples will still train format.")
if len(GOLD) < 8:
    raise RuntimeError("Run V6-03 through V6-03j before this cell.")


In [ ]:
# ============================================================
# V6-05 — SPLIT AND SAVE V6 JSONL
# ============================================================

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            payload = {"messages": row["messages"]}
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
    print("wrote", len(rows), "->", path)


n = len(v6_rows)
n_train = max(1, int(n * 0.8))
n_val = max(1, int(n * 0.1)) if n >= 10 else 1
train_rows = v6_rows[:n_train]
val_rows = v6_rows[n_train:n_train + n_val] or train_rows[:1]
test_rows = v6_rows[n_train + n_val:] or val_rows[:1]

# Keep at least one gold example in validation.
if gold_msgs and not any(
    r["messages"][-1]["content"] == gold_msgs[0]["messages"][-1]["content"]
    for r in val_rows
):
    val_rows = [gold_msgs[0]] + list(val_rows)

write_jsonl(V6_DATA_DIR / "train.jsonl", train_rows)
write_jsonl(V6_DATA_DIR / "val.jsonl", val_rows)
write_jsonl(V6_DATA_DIR / "test.jsonl", test_rows)
print("train/val/test:", len(train_rows), len(val_rows), len(test_rows))


In [ ]:
# ============================================================
# V6-06 — TOKENIZE (completion-only loss on assistant)
# Same masking approach as V5-10. Do not use rfind on raw assistant text.
# ============================================================

from datasets import Dataset
from transformers import AutoTokenizer

if "BASE_MODEL_NAME" not in globals():
    BASE_MODEL_NAME = "Qwen/Qwen3-4B"
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 2048


def tokenize_supervised_v6(example):
    messages = example["messages"]
    full_text = apply_chat(tokenizer, messages, add_generation_prompt=False)
    prompt_text = apply_chat(tokenizer, messages[:-1], add_generation_prompt=True)
    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False,
    )
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False,
    )
    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    prompt_len = min(len(prompt["input_ids"]), max(len(input_ids) - 1, 0))
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


def rows_to_dataset(rows):
    return Dataset.from_dict({"messages": [r["messages"] for r in rows]})


ds_train_raw = rows_to_dataset(train_rows)
ds_val_raw = rows_to_dataset(val_rows)
ds_train = ds_train_raw.map(
    tokenize_supervised_v6,
    remove_columns=ds_train_raw.column_names,
    desc="Tokenizing V6 train",
)
ds_val = ds_val_raw.map(
    tokenize_supervised_v6,
    remove_columns=ds_val_raw.column_names,
    desc="Tokenizing V6 val",
)

sample_labels = ds_train[0]["labels"]
n_supervised = sum(1 for x in sample_labels if x != -100)
print("tokenized train/val:", len(ds_train), len(ds_val))
print("sample tokens:", len(sample_labels), "supervised:", n_supervised)
if n_supervised < 8:
    raise RuntimeError("Completion mask looks wrong. Check chat template / enable_thinking.")


In [ ]:
# ============================================================
# V6-07 — 4-BIT MODEL + CONTINUE FROM V5 LORA IF PRESENT
# Set CONTINUE_FROM_V5 = False to train a fresh format adapter.
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

CONTINUE_FROM_V5 = True

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
else:
    raise RuntimeError("T4 GPU required. Runtime → Change runtime type → T4 GPU.")

install_gc_compat_patch()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

load_kwargs = dict(
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, attn_implementation="sdpa", **load_kwargs
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, **load_kwargs)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

v5_ok = (V5_ADAPTER_DIR / "adapter_config.json").exists()
if CONTINUE_FROM_V5 and v5_ok:
    print("Continuing from V5 adapter:", V5_ADAPTER_DIR)
    try:
        model = PeftModel.from_pretrained(model, str(V5_ADAPTER_DIR), is_trainable=True)
    except TypeError:
        model = PeftModel.from_pretrained(model, str(V5_ADAPTER_DIR))
        for name, param in model.named_parameters():
            if "lora" in name.lower():
                param.requires_grad = True
else:
    print("Training a fresh LoRA (no usable V5 adapter or CONTINUE_FROM_V5=False)")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)

model.train()
install_gc_compat_patch()
model.print_trainable_parameters()
print("✅ V6 QLoRA model ready")


In [ ]:
# ============================================================
# V6-07b — TRAINING GATE (fail fast before the long V6-08 run)
# ============================================================

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
labels0 = ds_train[0]["labels"]
supervised = sum(1 for x in labels0 if x != -100)

checks = {
    "GPU": torch.cuda.is_available(),
    "model": model is not None,
    "tokenizer": tokenizer is not None,
    "train rows": len(ds_train) > 0,
    "val rows": len(ds_val) > 0,
    "trainable LoRA params": trainable > 0,
    "completion labels": supervised > 0,
    "gold examples": len(GOLD) >= 8,
    "adapter dir parent": V6_ADAPTER_DIR.parent.exists(),
}

for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

if not all(checks.values()):
    raise RuntimeError("V6 training gate failed. Fix the items above before V6-08.")

print("\n🟢 GATE PASSED — next cell starts fine-tuning")
print("Trainable params:", f"{trainable:,}")
print("Train rows:", len(ds_train), "Val rows:", len(ds_val))


In [ ]:
# ============================================================
# V6-08 — TRAINER + START FINE-TUNE
# Set QUICK_SMOKE_TRAIN_V6=True first to verify the loop (~12 steps),
# then rerun this cell with False for the real format fine-tune.
# ============================================================

from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

QUICK_SMOKE_TRAIN_V6 = False

install_gc_compat_patch()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

_ta = dict(
    output_dir=str(V6_CHECKPOINT_DIR),
    num_train_epochs=1 if QUICK_SMOKE_TRAIN_V6 else 3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.04,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    lr_scheduler_type="cosine",
    seed=42,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_grad_norm=0.3,
    dataloader_num_workers=0,
)
if QUICK_SMOKE_TRAIN_V6:
    _ta["max_steps"] = 12
    _ta["eval_strategy"] = "steps"
    _ta["eval_steps"] = 6
    _ta["save_strategy"] = "steps"
    _ta["save_steps"] = 6
    _ta["load_best_model_at_end"] = False


def build_args(payload):
    try:
        return TrainingArguments(**payload)
    except TypeError as first:
        payload = dict(payload)
        if "eval_strategy" in payload:
            payload["evaluation_strategy"] = payload.pop("eval_strategy")
        try:
            return TrainingArguments(**payload)
        except TypeError:
            for key in [
                "gradient_checkpointing_kwargs",
                "load_best_model_at_end",
                "metric_for_best_model",
                "greater_is_better",
                "warmup_ratio",
                "max_grad_norm",
                "dataloader_num_workers",
            ]:
                payload.pop(key, None)
            payload["optim"] = "adamw_torch"
            print("TrainingArguments fallback:", first)
            return TrainingArguments(**payload)


args = build_args(_ta)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
)

print("QUICK_SMOKE_TRAIN_V6:", QUICK_SMOKE_TRAIN_V6)
print("epochs:", args.num_train_epochs)
print("STARTING V6 QLORA TRAINING — keep this Colab tab open.")
train_result = trainer.train()
print("\n🟢 V6 TRAINING COMPLETE")
print("global_step:", train_result.global_step)
print("training_loss:", getattr(train_result, "training_loss", None))


In [ ]:
# ============================================================
# V6-09 — SAVE ADAPTER + ZIP
# ============================================================

V6_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(V6_ADAPTER_DIR))
tokenizer.save_pretrained(str(V6_ADAPTER_DIR))

metrics = {
    "version": "v6",
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "adapter": str(V6_ADAPTER_DIR),
    "base_model": BASE_MODEL_NAME,
    "train_rows": len(train_rows),
    "val_rows": len(val_rows),
    "gold_unique": len(GOLD),
    "quick_smoke": QUICK_SMOKE_TRAIN_V6,
    "global_step": getattr(train_result, "global_step", None),
    "training_loss": getattr(train_result, "training_loss", None),
}
(V6_ADAPTER_DIR / "v6_train_meta.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
(PROJECT_ROOT / "configs" / "active_adapter_v6.json").write_text(
    json.dumps(
        {
            "active_adapter": "v6",
            "base_model": BASE_MODEL_NAME,
            "adapter_path": str(V6_ADAPTER_DIR),
            "updated": metrics["saved_at"],
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved adapter:", V6_ADAPTER_DIR)
for item in sorted(V6_ADAPTER_DIR.iterdir()):
    size = item.stat().st_size / 1024**2 if item.is_file() else 0
    print(f"  {item.name:40} {size:.2f} MB")

zip_path = PROJECT_ROOT / "adapters" / "ResumeOptimizer-LoRA-v6.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", V6_ADAPTER_DIR)
print("Zip:", zip_path, "MB:", round(zip_path.stat().st_size / 1024**2, 2))
if QUICK_SMOKE_TRAIN_V6:
    print("⚠️ Smoke run. Rerun V6-08 with QUICK_SMOKE_TRAIN_V6=False before shipping.")
